<a href="https://colab.research.google.com/github/NafishaNower/Job-Skill-Matching-System-using-NLP/blob/main/Skill_match_with_JD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

JD dataset prepare


import dataset from kaggle


This code downloads the Kaggle job description dataset and stores the local file path in the path variable.

In [1]:
import kagglehub

path = kagglehub.dataset_download("adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles")

print("Path to dataset files:", path)

100%|██████████| 203k/203k [00:00<00:00, 52.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/versions/1


The dataset is copied from Colab’s temporary storage to Google Drive to ensure persistence across sessions.

In [2]:
from google.colab import drive
import shutil
import os

source_dir = '/root/.cache/kagglehub/datasets/adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles/versions/1'
destination_parent_dir = '/content/drive/MyDrive/skill_match_with_JD'
destination_dir = os.path.join(destination_parent_dir, os.path.basename(source_dir))

# Create the destination parent directory if it doesn't exist
os.makedirs(destination_parent_dir, exist_ok=True)

if os.path.exists(destination_dir):
    print(f"Destination directory '{destination_dir}' already exists. Skipping copy.")
else:
    try:
        # Copy the entire directory from the source to the destination
        shutil.copytree(source_dir, destination_dir)
        print(f" Dataset successfully copied from '{source_dir}' to '{destination_dir}'")
    except Exception as e:
        print(f"Error copying dataset: {e}")

# Update the 'path' variable to point to the new location in Drive for future use
path = destination_dir
print(f"'path' variable updated to: {path}")


Destination directory '/content/drive/MyDrive/skill_match_with_JD/1' already exists. Skipping copy.
'path' variable updated to: /content/drive/MyDrive/skill_match_with_JD/1


import libraries

*nltk-- for basic NLP text processing

-->Tokenization (sentence → words)
-->Stopwords remove
-->Stemming / Lemmatization

*spacy-advance NLP

-->Named Entity Recognition (Name, Location, Date)
-->POS tagging
-->Dependency parsing

*sentence-transformers

-->Sentence embedding
-->Semantic search
-->Similarity check (meaning-wise)

*rapidfuzz

--> fuzzy matching if not exact match then similar closest match


In [3]:

!pip install pandas numpy
!pip install nltk spacy
!pip install sentence-transformers
!pip install rapidfuzz  # For fuzzy matching
!pip install openpyxl  # For Excel files
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ============================================
# STEP 2.1: Load the Dataset
# ============================================

def load_jd_dataset(file_path):
    """
    This function loads a dataset dynamically based on file type (CSV or Excel) and returns a pandas DataFrame.
    """
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Use CSV or Excel.")

    print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
    return df

# ============================================
# STEP 2.2: Inspect Dataset Structure
# ============================================

def inspect_dataset(df):
    """
    Perform comprehensive data inspection
    This function performs a comprehensive dataset inspection including shape, columns, data types, missing values, and sample records.
    """
    report = {
        'shape': df.shape,
        'columns': df.columns.tolist(),
        'dtypes': df.dtypes.to_dict(),
        'missing_values': df.isnull().sum().to_dict(),
        'sample_rows': df.head(3).to_dict('records')
    }

    print("=" * 60)
    print("DATASET INSPECTION REPORT")
    print("=" * 60)
    print(f"\nShape: {report['shape'][0]} rows × {report['shape'][1]} columns")
    print(f"\nColumns: {report['columns']}")
    print(f"\nMissing Values:")
    for col, count in report['missing_values'].items():
        if count > 0:
            print(f"   - {col}: {count} ({count/len(df)*100:.1f}%)")

    print("\nSample Rows:")
    for i, row in enumerate(report['sample_rows'], 1):
        print(f"\n--- Row {i} ---")
        for key, value in row.items():
            print(f"{key}: {str(value)[:100]}...")  # Show first 100 chars

    return report

# ============================================
# STEP 2.3: Validate Required Columns
# ============================================

def validate_columns(df, required_columns):
    """
    Check if all required columns exist
    This function validates whether all required columns exist in the dataset and raises an error if any are missing.
    """
    missing_cols = set(required_columns) - set(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    print(f"All required columns present: {required_columns}")

# ============================================
# USAGE EXAMPLE
# ============================================

df_jd = load_jd_dataset('/content/drive/MyDrive/skill_match_with_JD/1/job_dataset.csv')
REQUIRED_COLUMNS = [
    'JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience',
    'Skills', 'Responsibilities', 'Keywords'
]

validate_columns(df_jd, REQUIRED_COLUMNS)
inspection_report = inspect_dataset(df_jd)

Dataset loaded: 1068 rows, 7 columns
All required columns present: ['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']
DATASET INSPECTION REPORT

Shape: 1068 rows × 7 columns

Columns: ['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']

Missing Values:
   - Title: 1 (0.1%)

Sample Rows:

--- Row 1 ---
JobID: NET-F-001...
Title: .NET Developer...
ExperienceLevel: Fresher...
YearsOfExperience: 0-1...
Skills: C#; VB.NET basics; .NET Framework; .NET Core fundamentals; ASP.NET; MVC; HTML; CSS; JavaScript basic...
Responsibilities: Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; S...
Keywords: .NET; C#; ASP.NET MVC; Entity Framework; SQL Server; LINQ; Visual Studio; Unit Testing...

--- Row 2 ---
JobID: NET-F-002...
Title: .NET Developer...
ExperienceLevel: Fresher...
YearsOfExperience: 0-1...
Skills: C#; .NET Framework basics; ASP.NET; Razor; 

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# ============================================
# STEP 3.1: Text Cleaning Functions
# ============================================

def clean_text(text):
    """
    Clean and normalize text data

    Steps:
        1. Handle None/NaN values
        2. Remove HTML tags
        3. Remove special characters (keep alphanumeric, spaces, periods)
        4. Normalize whitespace
        5. Convert to lowercase

    Args:
        text (str): Raw text

    Returns:
        str: Cleaned text
    """
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'http\S+|www.\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z0-9\s\.\,\-\+\#]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    text = text.lower().strip()

    return text

def split_skills(skills_str, delimiter=';'):
    """
    Split skills string into list and clean each skill

    Args:
        skills_str (str): Semicolon-separated skills
        delimiter (str): Delimiter character

    Returns:
        list: List of cleaned skills
    """
    if pd.isna(skills_str):
        return []

    skills = str(skills_str).split(delimiter)

    # Clean each skill
    cleaned_skills = []
    for skill in skills:
        skill = skill.strip()
        skill = re.sub(r'\s+(basics?|fundamentals?|experience)$', '', skill, flags=re.IGNORECASE)
        if skill:
            cleaned_skills.append(skill)

    return cleaned_skills

def normalize_experience(exp_str):
    """
    Extract numeric experience value from text
    """
    if pd.isna(exp_str):
        return 0.0

    exp_str = str(exp_str).lower()

    if 'fresher' in exp_str or exp_str == '0':
        return 0.0

    numbers = re.findall(r'\d+', exp_str)

    if len(numbers) == 2:
        return (int(numbers[0]) + int(numbers[1])) / 2
    elif len(numbers) == 1:
        return float(numbers[0])
    else:
        return 0.0

# ============================================
# STEP 3.2: Apply Cleaning to Dataset
# ============================================

def preprocess_jd_dataset(df):
    """
    Apply all preprocessing steps to JD dataset
    """
    df_clean = df.copy()

    print("Starting data preprocessing...")

    df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_').str.lower()

    df_clean['job_id'] = df_clean['jobid'].astype(str).str.strip()

    df_clean['job_title'] = df_clean['title'].apply(clean_text)

    df_clean['raw_responsibilities'] = df_clean['responsibilities']
    df_clean['raw_skills'] = df_clean['skills']
    df_clean['raw_keywords'] = df_clean['keywords']

    df_clean['clean_responsibilities'] = df_clean['responsibilities'].apply(clean_text)
    df_clean['clean_keywords'] = df_clean['keywords'].apply(clean_text)

    df_clean['skills_list'] = df_clean['skills'].apply(split_skills)

    df_clean['experience_level'] = df_clean['experiencelevel'].astype(str)
    df_clean['years_experience_numeric'] = df_clean['yearsofexperience'].apply(normalize_experience)

    initial_count = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=['job_title', 'clean_responsibilities'], keep='first')
    removed = initial_count - len(df_clean)
    print(f" Removed {removed} duplicate job postings")

    df_clean = df_clean[df_clean['clean_responsibilities'].str.len() > 10]

    print(f" Preprocessing complete: {len(df_clean)} jobs ready")

    return df_clean

# ============================================
# USAGE EXAMPLE
# ============================================

df_jd_clean = preprocess_jd_dataset(df_jd)

print("\nSample cleaned row:")
sample = df_jd_clean.iloc[0]
print(f"Job ID: {sample['job_id']}")
print(f"Title: {sample['job_title']}")
print(f"Experience: {sample['years_experience_numeric']} years")
print(f"Skills: {sample['skills_list'][:5]}...")  # First 5 skills
print(f"Clean Text: {sample['clean_responsibilities'][:200]}...")

Starting data preprocessing...
 Removed 21 duplicate job postings
 Preprocessing complete: 1047 jobs ready

Sample cleaned row:
Job ID: NET-F-001
Title: .net developer
Experience: 0.5 years
Skills: ['C#', 'VB.NET', '.NET Framework', '.NET Core', 'ASP.NET']...
Clean Text: assist in coding and debugging applications learn and apply .net framework and core fundamentals support team in building asp.net mvc web applications write basic sql queries and work with entity fram...


In [ ]:
# ============================================
# DIAGNOSTIC: Check what's in your data
# ============================================

def diagnose_extraction_failure(df):
    """
    Find out WHY extraction is failing
    """
    print("="*60)
    print("EXTRACTION FAILURE DIAGNOSTIC")
    print("="*60)

    # Check a worst case
    worst_case = df.nsmallest(1, 'f1_score').iloc[0]

    print("\n🔍 EXAMINING WORST CASE:")
    print(f"Job: {worst_case['job_title']}")
    print(f"F1 Score: {worst_case['f1_score']:.2f}")

    print(f"\n📋 Ground Truth Skills ({len(worst_case['ground_truth_skills'])} skills):")
    print(f"   {worst_case['ground_truth_skills']}")

    print(f"\n🔍 Extracted Skills ({len(worst_case['normalized_skills'])} skills):")
    print(f"   {worst_case['normalized_skills']}")

    print("\n📄 RAW TEXT FIELDS:")
    print("-"*60)

    print(f"\n1️⃣ RAW RESPONSIBILITIES (first 500 chars):")
    print(f"   {str(worst_case.get('raw_responsibilities', 'MISSING'))[:500]}")

    print(f"\n2️⃣ CLEAN RESPONSIBILITIES (first 500 chars):")
    print(f"   {str(worst_case.get('clean_text', 'MISSING'))[:500]}")

    print(f"\n3️⃣ RAW KEYWORDS (first 200 chars):")
    print(f"   {str(worst_case.get('raw_keywords', 'MISSING'))[:200]}")

    print(f"\n4️⃣ RAW SKILLS:")
    print(f"   {str(worst_case.get('raw_skills', 'MISSING'))[:200]}")

    # Check if skills appear in text
    print("\n🎯 SKILL PRESENCE CHECK:")
    print("-"*60)

    text_combined = str(worst_case.get('clean_text', '')) + ' ' + str(worst_case.get('clean_keywords', ''))
    text_lower = text_combined.lower()

    for skill in worst_case['ground_truth_skills'][:10]:
        skill_lower = skill.lower()
        found = skill_lower in text_lower
        print(f"   '{skill}' → {'✅ FOUND' if found else '❌ NOT IN TEXT'}")

    # Check pattern matching
    print("\n🔍 PATTERN MATCHING TEST:")
    print("-"*60)

    # Test if patterns would match
    test_patterns = [
        (r'\bpython\b', 'Python'),
        (r'\bdjango\b', 'Django'),
        (r'\bflask\b', 'Flask'),
        (r'\bnode\.js\b', 'Node.js'),
        (r'\bexpress\.js\b', 'Express.js'),
        (r'\bmongodb\b', 'MongoDB'),
        (r'\bpostgresql\b', 'PostgreSQL'),
        (r'\bgit\b', 'Git'),
    ]

    import re
    for pattern, skill_name in test_patterns:
        matches = re.findall(pattern, text_lower, re.IGNORECASE)
        print(f"   Pattern '{pattern}' for {skill_name}: {matches if matches else '❌ NO MATCH'}")

# Run diagnostic
diagnose_extraction_failure(df_jd_final)

EXTRACTION FAILURE DIAGNOSTIC

🔍 EXAMINING WORST CASE:
Job: backend developer - entry level
F1 Score: 0.00

📋 Ground Truth Skills (13 skills):
   ['Node.js', 'Python', 'Java', 'Express.Js', 'Flask', 'Django', 'MySQL', 'PostgreSQL', 'MongoDB', 'Redis', 'REST API', 'Graphql', 'Git']

🔍 Extracted Skills (3 skills):
   ['SQL', 'Nosql', 'API Integration']

📄 RAW TEXT FIELDS:
------------------------------------------------------------

1️⃣ RAW RESPONSIBILITIES (first 500 chars):
   Assist in building server-side applications; Develop RESTful APIs; Work with SQL/NoSQL databases; Participate in testing and debugging; Learn cloud deployment and security fundamentals

2️⃣ CLEAN RESPONSIBILITIES (first 500 chars):
   assist in building server-side applications develop restful apis work with sql nosql databases participate in testing and debugging learn cloud deployment and security fundamentals

3️⃣ RAW KEYWORDS (first 200 chars):
   Backend Developer; Fresher; Server-Side Development; API Devel

In [ ]:
# ============================================
# EMERGENCY TEST: Extract from Skills column directly
# ============================================

def test_extraction_from_skills_column(df):
    """
    Test: What if we extract from the Skills column instead of Responsibilities?
    """
    print("\n🧪 TESTING: Extraction from Skills column")
    print("="*60)

    extractor = EnhancedSkillExtractor()

    # Extract from raw_skills column (where ground truth comes from)
    df['test_extracted'] = df['raw_skills'].apply(
        lambda x: extractor.extract_with_patterns(str(x)) if pd.notna(x) else []
    )

    # Normalize
    normalizer = ImprovedSkillNormalizer(df_ontology, ontology_lookup)
    df['test_normalized'] = df['test_extracted'].apply(
        lambda skills: normalizer.normalize_skills_list(skills)
    )

    # Calculate metrics
    test_metrics = []
    for _, row in df.iterrows():
        extracted_set = set([s.lower() for s in row['test_normalized']])
        gt_set = set([s.lower() for s in row['ground_truth_skills']])

        if len(extracted_set) == 0 and len(gt_set) == 0:
            f1 = 1.0
        elif len(extracted_set) == 0 or len(gt_set) == 0:
            f1 = 0.0
        else:
            tp = len(extracted_set & gt_set)
            precision = tp / len(extracted_set) if extracted_set else 0
            recall = tp / len(gt_set) if gt_set else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        test_metrics.append(f1)

    avg_f1 = np.mean(test_metrics)

    print(f"\n📊 TEST RESULTS (extracting from Skills column):")
    print(f"   Average F1: {avg_f1:.3f}")

    if avg_f1 > 0.5:
        print("\n✅ DIAGNOSIS: Your Responsibilities field is probably empty/generic!")
        print("   Solution: Use Skills column as extraction source for this dataset.")
    else:
        print("\n❓ Skills column extraction also failed.")
        print("   Need to check ontology aliases.")

    return avg_f1

# Run test
test_f1 = test_extraction_from_skills_column(df_jd_final)


🧪 TESTING: Extraction from Skills column
   Loading SBERT model...

📊 TEST RESULTS (extracting from Skills column):
   Average F1: 0.468

❓ Skills column extraction also failed.
   Need to check ontology aliases.


This cell implements a smart data preparation module that converts cleaned job descriptions into structured training-ready data. It parses ground truth skills, applies weak supervision, generates augmented text for NER training, builds a skill co-occurrence matrix, and learns title-to-skill mappings from the dataset.

In [6]:
# ============================================
# STEP 4: SMART DATA PREPARATION
# ============================================
#
# Strategy:
# 1. Parse Skills column → Ground Truth Labels
# 2. Extract weak signals from text
# 3. Create augmented training data for NER
# 4. Create skill co-occurrence matrix for recommendation
# ============================================

import re
import json
import random
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

class SmartDataPreparator:
    """
    Professional data preparation for skill extraction

    Handles the core problem:
    Skills are NOT in Responsibilities text

    Solution:
    1. Parse Skills column as ground truth
    2. Augment text with skills (creates NER training data)
    3. Build co-occurrence matrix (enables recommendation)
    """

    def __init__(self):
        self.skill_patterns = self._create_patterns()
        self.co_occurrence_matrix = defaultdict(Counter)
        self.title_to_skills = defaultdict(list)

    def _create_patterns(self):
        """Extraction patterns for text mining"""
        return {
            'languages': [
                r'\b(python|java|javascript|typescript|c\+\+|c#|kotlin|swift|php|ruby|go)\b',
            ],
            'backend': [
                r'\b(node\.?js|express\.?js|django|flask|fastapi|spring\s*boot)\b',
                r'\b(rest\s*api|restful|graphql|grpc)\b',
            ],
            'frontend': [
                r'\b(react\.?js|reactjs|angular|vue\.?js|html5?|css3?|bootstrap)\b',
            ],
            'databases': [
                r'\b(mysql|postgresql|postgres|mongodb|redis|sqlite|sql\s+server)\b',
                r'\b(oracle|dynamodb|cassandra|firebase)\b',
            ],
            'tools': [
                r'\b(git|github|docker|kubernetes|jenkins|jira|postman)\b',
            ],
            'cloud': [
                r'\b(aws|azure|gcp|google\s+cloud)\b',
            ],
            'ml': [
                r'\b(tensorflow|pytorch|keras|scikit[\-\s]learn|pandas|numpy)\b',
                r'\b(machine\s+learning|deep\s+learning|nlp|computer\s+vision)\b',
            ],
            'mobile': [
                r'\b(android|kotlin|swift|flutter|react\s+native|xamarin)\b',
            ],
            'concepts': [
                r'\b(oop|agile|scrum|ci/?cd|microservices|tdd|unit\s+testing)\b',
                r'\b(design\s+patterns|solid\s+principles)\b',
            ],
            'dotnet': [
                r'\b(\.net|asp\.net|c#|entity\s+framework|linq|blazor)\b',
            ],
            'arvr': [
                r'\b(unity|unreal\s+engine|arkit|arcore|vuforia)\b',
            ],
        }

    def parse_ground_truth(self, skills_str):
        """
        Parse Skills column into clean list
        This is GROUND TRUTH
        """
        if pd.isna(skills_str) or str(skills_str).strip() == '':
            return []

        skills = re.split(r'[;,|]', str(skills_str))

        cleaned = []
        for skill in skills:
            skill = skill.strip()

            skill = re.sub(r'\([^)]+\)', '', skill).strip()

            skill = re.sub(
                r'\s+(basics?|fundamentals?|advanced?|expert|intermediate'
                r'|experience|expertise|proficiency|knowledge)$',
                '', skill, flags=re.IGNORECASE
            ).strip()

            skill = re.sub(
                r'^(basic|advanced?|deep|strong|expert|good|excellent)\s+',
                '', skill, flags=re.IGNORECASE
            ).strip()

            if skill and len(skill) > 1:
                cleaned.append(skill)

        return cleaned

    def extract_weak_labels_from_text(self, row):
        """
        Extract what patterns can find in text
        These are WEAK LABELS - honest signal from text
        """
        all_text = ' '.join([
            str(row.get('clean_text', '') or ''),
            str(row.get('clean_keywords', '') or ''),
            str(row.get('job_title', '') or ''),
        ]).lower()

        extracted = []
        for category, patterns in self.skill_patterns.items():
            for pattern in patterns:
                matches = re.findall(pattern, all_text, re.IGNORECASE)
                for match in matches:
                    if isinstance(match, tuple):
                        match = match[0]
                    match = match.strip()
                    if len(match) > 1:
                        extracted.append(match)

        return list(set(extracted))

    def create_augmented_text(self, responsibilities, skills_list):
        """
        AUGMENTATION TECHNIQUE:
        Inject skills into responsibilities text
        This creates synthetic training data for NER model
        """
        if not skills_list:
            return str(responsibilities)

        responsibilities = str(responsibilities) if pd.notna(responsibilities) else ''


        augmentation_sentences = []


        prog_langs = [s for s in skills_list if s.lower() in [
            'python', 'java', 'javascript', 'kotlin', 'swift', 'c#', 'c++'
        ]]
        frameworks = [s for s in skills_list if s.lower() in [
            'django', 'flask', 'node.js', 'react', 'angular', 'spring boot',
            'express.js', 'fastapi', '.net framework', 'asp.net'
        ]]
        databases = [s for s in skills_list if s.lower() in [
            'mysql', 'postgresql', 'mongodb', 'redis', 'sqlite', 'oracle'
        ]]
        remaining = [s for s in skills_list
                    if s not in prog_langs + frameworks + databases]


        if prog_langs:
            langs_text = ', '.join(prog_langs[:3])
            augmentation_sentences.append(
                f"Programming languages include {langs_text}."
            )

        if frameworks:
            frameworks_text = ', '.join(frameworks[:3])
            augmentation_sentences.append(
                f"Frameworks and technologies: {frameworks_text}."
            )

        if databases:
            db_text = ', '.join(databases[:3])
            augmentation_sentences.append(
                f"Database technologies: {db_text}."
            )

        if remaining:
            remaining_text = ', '.join(remaining[:4])
            augmentation_sentences.append(
                f"Additional skills: {remaining_text}."
            )


        augmented = ' '.join(augmentation_sentences)
        if responsibilities:
            augmented = augmented + ' ' + responsibilities

        return augmented

    def build_cooccurrence_matrix(self, df):
        """
        Build skill co-occurrence matrix
        This enables RECOMMENDATION:
        """
        print("   Building skill co-occurrence matrix...")

        for _, row in df.iterrows():
            skills = row.get('ground_truth_skills', [])


            for i, skill_a in enumerate(skills):
                for skill_b in skills:
                    if skill_a != skill_b:
                        self.co_occurrence_matrix[skill_a.lower()][skill_b.lower()] += 1

        print(f"Co-occurrence matrix: {len(self.co_occurrence_matrix)} unique skills")

        return self.co_occurrence_matrix

    def build_title_skill_map(self, df):
        """
        Build job title → skills mapping from data
        """
        print("   Building title→skill mapping from data...")

        title_skills = defaultdict(list)

        for _, row in df.iterrows():
            title = str(row.get('job_title', '')).lower().strip()
            skills = row.get('ground_truth_skills', [])

            if title and skills:
                title_skills[title].extend(skills)


        self.title_to_skills = {}
        for title, skills in title_skills.items():
            skill_counts = Counter(skills)
            total = len(df[df['job_title'].str.lower() == title])
            self.title_to_skills[title] = [
                skill for skill, count in skill_counts.most_common(15)
                if count >= max(1, total * 0.3)
            ]

        print(f"Title map: {len(self.title_to_skills)} unique job titles")

        return self.title_to_skills

    def create_ner_training_data(self, row):
        """
        Create NER training example with BIO tags
        """
        augmented_text = row.get('augmented_text', '')
        ground_truth_skills = row.get('ground_truth_skills', [])

        if not augmented_text or not ground_truth_skills:
            return None

        words = augmented_text.split()
        bio_tags = ['O'] * len(words)

        for skill in ground_truth_skills:
            skill_words = skill.lower().split()
            skill_len = len(skill_words)

            for i in range(len(words) - skill_len + 1):
                window = [w.lower().strip('.,;:') for w in words[i:i+skill_len]]

                if window == skill_words:
                    bio_tags[i] = 'B-SKILL'
                    for j in range(1, skill_len):
                        bio_tags[i+j] = 'I-SKILL'

        return {
            'text': augmented_text,
            'tokens': words,
            'bio_tags': bio_tags,
            'skills': ground_truth_skills
        }

    def process_dataset(self, df):
        """
        Run complete data preparation pipeline
        """
        print("\n Running smart data preparation...")
        print("="*60)

        # Step A: Parse ground truth
        print("\n[A] Parsing ground truth skills...")
        df['ground_truth_skills'] = df['raw_skills'].apply(
            self.parse_ground_truth
        )

        # Step B: Extract weak labels
        print("[B] Extracting weak labels from text...")
        df['weak_labels'] = df.apply(
            self.extract_weak_labels_from_text, axis=1
        )

        # Step C: Create augmented text
        print("[C] Creating augmented training text...")
        df['augmented_text'] = df.apply(
            lambda row: self.create_augmented_text(
                row.get('clean_text', ''),
                row['ground_truth_skills']
            ), axis=1
        )

        # Step D: Build co-occurrence matrix
        print("[D] Building co-occurrence matrix...")
        self.build_cooccurrence_matrix(df)

        # Step E: Build title→skill map from data
        print("[E] Building title→skill map...")
        self.build_title_skill_map(df)

        # Step F: Create NER training examples
        print("[F] Creating NER training examples...")
        df['ner_training'] = df.apply(
            self.create_ner_training_data, axis=1
        )

        ner_valid = df['ner_training'].notna().sum()
        print(f"\nData preparation complete:")
        print(f"   Total jobs:           {len(df)}")
        print(f"   Augmented examples:   {len(df)}")
        print(f"   NER training ready:   {ner_valid}")

        avg_gt = df['ground_truth_skills'].apply(len).mean()
        avg_weak = df['weak_labels'].apply(len).mean()
        coverage = avg_weak / avg_gt * 100 if avg_gt > 0 else 0

        print(f"   Avg GT skills/job:    {avg_gt:.1f}")
        print(f"   Avg weak labels/job:  {avg_weak:.1f}")
        print(f"   Text coverage:        {coverage:.1f}%")

        return df, self.co_occurrence_matrix, self.title_to_skills


# ============================================
# USAGE
# ============================================

preparator = SmartDataPreparator()
df_jd_clean, co_occurrence, title_skill_map = preparator.process_dataset(df_jd_clean)


 Running smart data preparation...

[A] Parsing ground truth skills...
[B] Extracting weak labels from text...
[C] Creating augmented training text...
[D] Building co-occurrence matrix...
   Building skill co-occurrence matrix...
Co-occurrence matrix: 1706 unique skills
[E] Building title→skill map...
   Building title→skill mapping from data...
Title map: 218 unique job titles
[F] Creating NER training examples...

Data preparation complete:
   Total jobs:           1047
   Augmented examples:   1047
   NER training ready:   1047
   Avg GT skills/job:    12.4
   Avg weak labels/job:  1.7
   Text coverage:        13.8%


This cell implements a production-grade skill normalization system using a structured ontology with aliases, fuzzy string matching, and semantic similarity via SBERT. It standardizes extracted and ground truth skills into canonical representations to ensure consistent downstream matching and recommendation performance.

In [7]:
# ============================================
# STEP 5: PROFESSIONAL NORMALIZATION
# ============================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz, process

def create_production_ontology():
    """
    ontology with comprehensive aliases
    """

    ontology_data = [
        # Languages
        {'skill_id': 'S001', 'skill_name': 'Python', 'aliases': 'python,python3,py,python programming', 'category': 'Language'},
        {'skill_id': 'S002', 'skill_name': 'Java', 'aliases': 'java,java8,java11,java17', 'category': 'Language'},
        {'skill_id': 'S003', 'skill_name': 'JavaScript', 'aliases': 'javascript,js,es6,ecmascript', 'category': 'Language'},
        {'skill_id': 'S004', 'skill_name': 'TypeScript', 'aliases': 'typescript,ts', 'category': 'Language'},
        {'skill_id': 'S005', 'skill_name': 'C#', 'aliases': 'c#,csharp,c sharp', 'category': 'Language'},
        {'skill_id': 'S006', 'skill_name': 'C++', 'aliases': 'c++,cpp,cplusplus', 'category': 'Language'},
        {'skill_id': 'S007', 'skill_name': 'Kotlin', 'aliases': 'kotlin', 'category': 'Language'},
        {'skill_id': 'S008', 'skill_name': 'Swift', 'aliases': 'swift', 'category': 'Language'},
        {'skill_id': 'S009', 'skill_name': 'PHP', 'aliases': 'php,php7,php8', 'category': 'Language'},
        {'skill_id': 'S010', 'skill_name': 'Ruby', 'aliases': 'ruby', 'category': 'Language'},
        {'skill_id': 'S011', 'skill_name': 'Go', 'aliases': 'go,golang', 'category': 'Language'},

        # Backend Frameworks - EXPANDED ALIASES
        {'skill_id': 'S020', 'skill_name': 'Node.js', 'aliases': 'node.js,nodejs,node,node js', 'category': 'Framework'},
        {'skill_id': 'S021', 'skill_name': 'Express.js', 'aliases': 'express.js,expressjs,express,express js', 'category': 'Framework'},
        {'skill_id': 'S022', 'skill_name': 'Django', 'aliases': 'django', 'category': 'Framework'},
        {'skill_id': 'S023', 'skill_name': 'Flask', 'aliases': 'flask', 'category': 'Framework'},
        {'skill_id': 'S024', 'skill_name': 'FastAPI', 'aliases': 'fastapi,fast api', 'category': 'Framework'},
        {'skill_id': 'S025', 'skill_name': 'Spring Boot', 'aliases': 'spring boot,spring,springboot', 'category': 'Framework'},

        # Frontend Frameworks
        {'skill_id': 'S030', 'skill_name': 'React', 'aliases': 'react,reactjs,react.js,react js', 'category': 'Framework'},
        {'skill_id': 'S031', 'skill_name': 'Angular', 'aliases': 'angular,angularjs,angular.js', 'category': 'Framework'},
        {'skill_id': 'S032', 'skill_name': 'Vue', 'aliases': 'vue,vuejs,vue.js,vue js', 'category': 'Framework'},
        {'skill_id': 'S033', 'skill_name': 'Next.js', 'aliases': 'next.js,nextjs,next,next js', 'category': 'Framework'},

        # .NET
        {'skill_id': 'S040', 'skill_name': '.NET Framework', 'aliases': '.net framework,.net,.net fw,dotnet', 'category': 'Framework'},
        {'skill_id': 'S041', 'skill_name': '.NET Core', 'aliases': '.net core,dotnet core', 'category': 'Framework'},
        {'skill_id': 'S042', 'skill_name': 'ASP.NET', 'aliases': 'asp.net,asp,aspnet', 'category': 'Framework'},
        {'skill_id': 'S043', 'skill_name': 'ASP.NET MVC', 'aliases': 'asp.net mvc,mvc,aspnet mvc', 'category': 'Framework'},
        {'skill_id': 'S044', 'skill_name': 'Entity Framework', 'aliases': 'entity framework,ef,ef core', 'category': 'Framework'},
        {'skill_id': 'S045', 'skill_name': 'LINQ', 'aliases': 'linq', 'category': 'Framework'},

        # Databases - EXPANDED
        {'skill_id': 'S050', 'skill_name': 'MySQL', 'aliases': 'mysql,my sql', 'category': 'Database'},
        {'skill_id': 'S051', 'skill_name': 'PostgreSQL', 'aliases': 'postgresql,postgres,psql', 'category': 'Database'},
        {'skill_id': 'S052', 'skill_name': 'MongoDB', 'aliases': 'mongodb,mongo', 'category': 'Database'},
        {'skill_id': 'S053', 'skill_name': 'Redis', 'aliases': 'redis', 'category': 'Database'},
        {'skill_id': 'S054', 'skill_name': 'SQL Server', 'aliases': 'sql server,mssql,microsoft sql,ms sql', 'category': 'Database'},
        {'skill_id': 'S055', 'skill_name': 'Oracle', 'aliases': 'oracle,oracle db', 'category': 'Database'},
        {'skill_id': 'S056', 'skill_name': 'SQLite', 'aliases': 'sqlite,sql lite', 'category': 'Database'},
        {'skill_id': 'S057', 'skill_name': 'Cassandra', 'aliases': 'cassandra', 'category': 'Database'},

        # SQL Concepts
        {'skill_id': 'S060', 'skill_name': 'SQL', 'aliases': 'sql,structured query language,t-sql,tsql', 'category': 'Database'},
        {'skill_id': 'S061', 'skill_name': 'NoSQL', 'aliases': 'nosql,no sql', 'category': 'Database'},

        # APIs
        {'skill_id': 'S070', 'skill_name': 'REST API', 'aliases': 'rest,rest api,restful,restful api,restful apis', 'category': 'API'},
        {'skill_id': 'S071', 'skill_name': 'GraphQL', 'aliases': 'graphql,graph ql,graphql basics', 'category': 'API'},
        {'skill_id': 'S072', 'skill_name': 'API Integration', 'aliases': 'api integration,api development,api', 'category': 'API'},

        # ML/AI
        {'skill_id': 'S080', 'skill_name': 'Machine Learning', 'aliases': 'machine learning,ml,machinelearning', 'category': 'ML/AI'},
        {'skill_id': 'S081', 'skill_name': 'Deep Learning', 'aliases': 'deep learning,dl', 'category': 'ML/AI'},
        {'skill_id': 'S082', 'skill_name': 'TensorFlow', 'aliases': 'tensorflow,tf', 'category': 'ML/AI'},
        {'skill_id': 'S083', 'skill_name': 'PyTorch', 'aliases': 'pytorch,torch', 'category': 'ML/AI'},
        {'skill_id': 'S084', 'skill_name': 'Keras', 'aliases': 'keras', 'category': 'ML/AI'},
        {'skill_id': 'S085', 'skill_name': 'Scikit-learn', 'aliases': 'scikit-learn,sklearn,scikit learn', 'category': 'ML/AI'},
        {'skill_id': 'S086', 'skill_name': 'NLP', 'aliases': 'nlp,natural language processing', 'category': 'ML/AI'},
        {'skill_id': 'S087', 'skill_name': 'Computer Vision', 'aliases': 'computer vision,cv', 'category': 'ML/AI'},

        # Data Science
        {'skill_id': 'S090', 'skill_name': 'Pandas', 'aliases': 'pandas', 'category': 'Data'},
        {'skill_id': 'S091', 'skill_name': 'NumPy', 'aliases': 'numpy', 'category': 'Data'},
        {'skill_id': 'S092', 'skill_name': 'Data Science', 'aliases': 'data science,data analysis', 'category': 'Data'},

        # Tools
        {'skill_id': 'S100', 'skill_name': 'Git', 'aliases': 'git,version control', 'category': 'Tool'},
        {'skill_id': 'S101', 'skill_name': 'GitHub', 'aliases': 'github', 'category': 'Tool'},
        {'skill_id': 'S102', 'skill_name': 'Docker', 'aliases': 'docker,containerization', 'category': 'Tool'},
        {'skill_id': 'S103', 'skill_name': 'Kubernetes', 'aliases': 'kubernetes,k8s', 'category': 'Tool'},
        {'skill_id': 'S104', 'skill_name': 'Visual Studio', 'aliases': 'visual studio,vs', 'category': 'Tool'},
        {'skill_id': 'S105', 'skill_name': 'VS Code', 'aliases': 'vs code,vscode', 'category': 'Tool'},
        {'skill_id': 'S106', 'skill_name': 'Android Studio', 'aliases': 'android studio', 'category': 'Tool'},
        {'skill_id': 'S107', 'skill_name': 'JIRA', 'aliases': 'jira', 'category': 'Tool'},
        {'skill_id': 'S108', 'skill_name': 'Postman', 'aliases': 'postman', 'category': 'Tool'},

        # Mobile
        {'skill_id': 'S110', 'skill_name': 'Android Development', 'aliases': 'android,android development', 'category': 'Mobile'},
        {'skill_id': 'S111', 'skill_name': 'iOS Development', 'aliases': 'ios,ios development', 'category': 'Mobile'},
        {'skill_id': 'S112', 'skill_name': 'React Native', 'aliases': 'react native,react-native', 'category': 'Mobile'},
        {'skill_id': 'S113', 'skill_name': 'Flutter', 'aliases': 'flutter', 'category': 'Mobile'},

        # AR/VR
        {'skill_id': 'S120', 'skill_name': 'Unity', 'aliases': 'unity,unity engine,unity 3d', 'category': 'AR/VR'},
        {'skill_id': 'S121', 'skill_name': 'Unreal Engine', 'aliases': 'unreal,unreal engine', 'category': 'AR/VR'},
        {'skill_id': 'S122', 'skill_name': 'ARKit', 'aliases': 'arkit', 'category': 'AR/VR'},
        {'skill_id': 'S123', 'skill_name': 'ARCore', 'aliases': 'arcore', 'category': 'AR/VR'},
        {'skill_id': 'S124', 'skill_name': 'Vuforia', 'aliases': 'vuforia', 'category': 'AR/VR'},

        # Cloud
        {'skill_id': 'S130', 'skill_name': 'AWS', 'aliases': 'aws,amazon web services', 'category': 'Cloud'},
        {'skill_id': 'S131', 'skill_name': 'Azure', 'aliases': 'azure,microsoft azure', 'category': 'Cloud'},
        {'skill_id': 'S132', 'skill_name': 'Google Cloud', 'aliases': 'gcp,google cloud', 'category': 'Cloud'},
        {'skill_id': 'S133', 'skill_name': 'Cloud', 'aliases': 'cloud,cloud computing', 'category': 'Cloud'},

        # Concepts
        {'skill_id': 'S140', 'skill_name': 'OOP', 'aliases': 'oop,object oriented programming,object-oriented', 'category': 'Concept'},
        {'skill_id': 'S141', 'skill_name': 'Agile', 'aliases': 'agile,scrum,agile methodologies', 'category': 'Concept'},
        {'skill_id': 'S142', 'skill_name': 'CI/CD', 'aliases': 'ci/cd,continuous integration,devops', 'category': 'Concept'},
        {'skill_id': 'S143', 'skill_name': 'Unit Testing', 'aliases': 'unit testing,testing,tdd', 'category': 'Concept'},
        {'skill_id': 'S144', 'skill_name': 'Microservices', 'aliases': 'microservices', 'category': 'Concept'},
        {'skill_id': 'S145', 'skill_name': 'UI/UX Design', 'aliases': 'ui/ux,ui design,ux design,user interface', 'category': 'Design'},

        # Web
        {'skill_id': 'S150', 'skill_name': 'HTML', 'aliases': 'html,html5', 'category': 'Web'},
        {'skill_id': 'S151', 'skill_name': 'CSS', 'aliases': 'css,css3', 'category': 'Web'},
        {'skill_id': 'S152', 'skill_name': 'XML', 'aliases': 'xml', 'category': 'Web'},
        {'skill_id': 'S153', 'skill_name': 'Ajax', 'aliases': 'ajax', 'category': 'Web'},
    ]

    df_ontology = pd.DataFrame(ontology_data)

    ontology_lookup = {}
    for _, row in df_ontology.iterrows():
        canonical = row['skill_name']
        aliases = row['aliases'].split(',')

        for alias in aliases:
            ontology_lookup[alias.lower().strip()] = canonical

    print(f" Production ontology: {len(df_ontology)} skills, {len(ontology_lookup)} aliases")

    return df_ontology, ontology_lookup


class ProductionNormalizer:
    """Production-grade normalizer with strict matching"""

    def __init__(self, ontology_df, ontology_lookup):
        self.ontology_df = ontology_df
        self.ontology_lookup = ontology_lookup

        print("   Loading SBERT model...")
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

        canonical_skills = ontology_df['skill_name'].tolist()
        self.canonical_embeddings = self.sbert_model.encode(canonical_skills)
        self.canonical_skills = canonical_skills

    def normalize_single_skill(self, skill):
        """Strict 3-tier normalization"""
        skill_lower = skill.lower().strip()

        # STEP 1: Exact match (90% of cases)
        if skill_lower in self.ontology_lookup:
            return self.ontology_lookup[skill_lower]

        # STEP 2: Fuzzy match with HIGH threshold (92%)
        fuzzy_match = process.extractOne(
            skill_lower,
            self.ontology_lookup.keys(),
            scorer=fuzz.ratio,
            score_cutoff=92
        )

        if fuzzy_match:
            return self.ontology_lookup[fuzzy_match[0]]

        # STEP 3: Semantic match (0.88+)
        skill_embedding = self.sbert_model.encode([skill])
        similarities = cosine_similarity(skill_embedding, self.canonical_embeddings)[0]

        max_idx = similarities.argmax()
        max_score = similarities[max_idx]

        if max_score >= 0.88:
            return self.canonical_skills[max_idx]

        # STEP 4: Return original (prevents hallucinations)
        return skill.title()

    def normalize_skills_list(self, skills):
        """Normalize list with deduplication"""
        normalized = []
        seen = set()

        for skill in skills:
            norm_skill = self.normalize_single_skill(skill)

            if norm_skill.lower() not in seen:
                seen.add(norm_skill.lower())
                normalized.append(norm_skill)

        return normalized


def apply_production_normalization(df, ontology_df, ontology_lookup):
    """Apply normalization"""
    print("\nApplying production normalization...")

    normalizer = ProductionNormalizer(ontology_df, ontology_lookup)

    print("   Normalizing ground truth skills...")
    df['ground_truth_normalized'] = df['ground_truth_skills'].apply(
        lambda skills: normalizer.normalize_skills_list(skills)
    )

    print("   Normalizing extracted skills...")
    df['normalized_skills'] = df['extracted_skills'].apply(
        lambda skills: normalizer.normalize_skills_list(skills)
    )

    print("Normalization complete")

    print(f"\nNormalization example:")
    sample = df.iloc[0]
    print(f"Ground Truth:  {sample['ground_truth_skills'][:5]}")
    print(f"Normalized:    {sample['ground_truth_normalized'][:5]}")

    return df

# ============================================
# USAGE
# ============================================



df_ontology, ontology_lookup = create_production_ontology()

normalizer = ProductionNormalizer(df_ontology, ontology_lookup)

df_jd_clean['ground_truth_normalized'] = df_jd_clean['ground_truth_skills'].apply(
    lambda skills: normalizer.normalize_skills_list(skills)
)

df_jd_clean['weak_labels_normalized'] = df_jd_clean['weak_labels'].apply(
    lambda skills: normalizer.normalize_skills_list(skills)
)

print("Normalization complete")

 Production ontology: 83 skills, 189 aliases
   Loading SBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Normalization complete


This cell constructs the final structured dataset for modeling and production use. It consolidates job metadata, cleaned and augmented text, normalized ground-truth skills, weak labels, NER training data, and metadata such as skill count and processing date into a single professional DataFrame.

In [8]:
# ============================================
# STEP 6: FINAL DATASET
# ============================================

def create_final_dataset(df, co_occurrence, title_skill_map):
    """
    Create final professional dataset
    """
    print("Creating final dataset...")

    final_df = pd.DataFrame({
        # Identity
        'job_id':                df['job_id'],
        'job_title':             df['job_title'],
        'experience_level':      df['experience_level'],
        'years_experience':      df['years_experience_numeric'],

        # Text fields
        'raw_responsibilities':  df['raw_responsibilities'],
        'clean_text':            df.get('clean_text', df.get('clean_responsibilities', '')),
        'augmented_text':        df['augmented_text'],

        # Skills
        'ground_truth_skills':   df['ground_truth_normalized'],
        'weak_labels':           df['weak_labels_normalized'],

        # Training data
        'ner_training':          df['ner_training'],

        # Metadata
        'skill_count':           df['ground_truth_normalized'].apply(len),
        'processed_date':        pd.Timestamp.now().strftime('%Y-%m-%d'),
    })

    print(f"Final dataset: {len(final_df)} jobs")
    return final_df

# ============================================
# USAGE
# ============================================

df_jd_final = create_final_dataset(
    df_jd_clean, co_occurrence, title_skill_map
)

Creating final dataset...
Final dataset: 1047 jobs


This cell performs a multi-layer evaluation of three approaches: text-based extraction, skill recommendation, and a hybrid system. It computes precision, recall, and F1-score on a held-out test set, analyzes per-job-title performance, and reports dataset limitations such as text skill coverage. This ensures honest and research-grade evaluation of system performance.

In [9]:
# ============================================
# STEP 7: HONEST MULTI-LAYER EVALUATION
# ============================================

from sklearn.model_selection import train_test_split

def evaluate_all_approaches(df, preparator):
    """
    Layer 1: Text extraction only (baseline)
    Layer 2: Recommendation from title + co-occurrence
    Layer 3: Combined hybrid system
    """
    print("MULTI-LAYER HONEST EVALUATION")
    print("="*60)

    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42
    )

    print(f"\nDataset Split:")
    print(f"   Train: {len(train_df)} | Test: {len(test_df)}")

    # ============================================
    # LAYER 1: Text Extraction Baseline
    # ============================================

    print("\n[Layer 1] Text extraction baseline...")

    layer1_metrics = compute_metrics(
        test_df['weak_labels'].tolist(),
        test_df['ground_truth_skills'].tolist()
    )

    # ============================================
    # LAYER 2: Skill Recommendation
    # ============================================

    print("[Layer 2] Skill recommendation...")

    def recommend_skills(row, title_skill_map, co_occurrence):
        """
        Recommend skills using:
        1. Title-based lookup (learned from training data)
        2. Co-occurrence expansion
        """
        title = str(row.get('job_title', '')).lower().strip()
        weak = [s.lower() for s in row.get('weak_labels', [])]

        recommended = []

        # Title-based recommendation
        if title in title_skill_map:
            recommended.extend(title_skill_map[title][:8])

        # Co-occurrence expansion
        for skill in weak:
            if skill in co_occurrence:
                top_cooccurring = [
                    s for s, count in co_occurrence[skill].most_common(3)
                    if count > 2
                ]
                recommended.extend(top_cooccurring)

        return list(set(recommended))

    test_df = test_df.copy()
    test_df['recommended'] = test_df.apply(
        lambda row: recommend_skills(
            row, preparator.title_to_skills, preparator.co_occurrence_matrix
        ), axis=1
    )

    layer2_metrics = compute_metrics(
        test_df['recommended'].tolist(),
        test_df['ground_truth_skills'].tolist()
    )

    # ============================================
    # LAYER 3: Combined Hybrid System
    # ============================================

    print("[Layer 3] Combined hybrid system...")

    test_df['combined'] = test_df.apply(
        lambda row: list(set(
            row.get('weak_labels', []) +
            row.get('recommended', [])
        )), axis=1
    )

    layer3_metrics = compute_metrics(
        test_df['combined'].tolist(),
        test_df['ground_truth_skills'].tolist()
    )

    # ============================================
    # PRINT RESULTS TABLE
    # ============================================

    print("\n" + "="*65)
    print("HONEST EVALUATION RESULTS")
    print("="*65)
    print(f"{'Approach':<35} {'Precision':>9} {'Recall':>9} {'F1':>9}")
    print("-"*65)

    approaches = {
        'Layer 1: Text Extraction':      layer1_metrics,
        'Layer 2: Recommendation':       layer2_metrics,
        'Layer 3: Hybrid (Combined)':    layer3_metrics,
    }

    for name, metrics in approaches.items():
        print(f"{name:<35} {metrics['precision']:>9.3f} "
              f"{metrics['recall']:>9.3f} {metrics['f1']:>9.3f}")

    print("-"*65)

    # ============================================
    # IMPROVEMENT ANALYSIS
    # ============================================

    print(f"\nIMPROVEMENT ANALYSIS:")
    base_f1   = layer1_metrics['f1']
    rec_f1    = layer2_metrics['f1']
    hybrid_f1 = layer3_metrics['f1']

    print(f"   Adding Recommendation improved F1 by: "
          f"+{(rec_f1 - base_f1):.3f} "
          f"({(rec_f1-base_f1)/max(base_f1,0.001)*100:.0f}%)")

    print(f"   Hybrid system improved F1 by:         "
          f"+{(hybrid_f1 - base_f1):.3f} "
          f"({(hybrid_f1-base_f1)/max(base_f1,0.001)*100:.0f}%)")

    # ============================================
    # PER-TITLE ANALYSIS
    # ============================================

    print(f"\nPER JOB TITLE F1 (Test Set):")
    print("-"*65)

    test_df['f1'] = test_df.apply(
        lambda row: compute_row_f1(
            row['combined'],
            row['ground_truth_skills']
        ), axis=1
    )

    title_f1 = (test_df.groupby('job_title')['f1']
                .mean()
                .sort_values(ascending=False))

    print("\nBest performing roles:")
    for title, f1 in title_f1.head(5).items():
        bar = '█' * int(f1 * 20)
        print(f"   {title:<35} {bar:<20} {f1:.2f}")

    print("\nWorst performing roles:")
    for title, f1 in title_f1.tail(5).items():
        bar = '█' * int(f1 * 20)
        print(f"   {title:<35} {bar:<20} {f1:.2f}")

    # ============================================
    # DATASET LIMITATION REPORT
    # ============================================

    text_coverage = (
        test_df['weak_labels'].apply(len) /
        test_df['ground_truth_skills'].apply(lambda x: max(len(x), 1))
    ).mean()

    print(f"\n📋 DATASET ANALYSIS:")
    print(f"   Text coverage of skills:  {text_coverage*100:.1f}%")
    print(f"   Skills found in text:     {text_coverage*100:.1f}%")
    print(f"   Skills only in Skills col:{(1-text_coverage)*100:.1f}%")

    if text_coverage < 0.3:
        print(f"\n   Low text coverage is expected in structured datasets")
        print(f"    Recommendation system compensates for this gap")
        print(f"    NER model on augmented data will improve further")

    return approaches, test_df


def compute_metrics(extracted_list, ground_truth_list):
    """Compute precision, recall, F1 across dataset"""
    all_p, all_r, all_f1 = [], [], []

    for extracted, ground_truth in zip(extracted_list, ground_truth_list):
        if not isinstance(extracted, list):
            extracted = []
        if not isinstance(ground_truth, list):
            ground_truth = []

        ext_set = set([s.lower().strip() for s in extracted])
        gt_set  = set([s.lower().strip() for s in ground_truth])

        if not ext_set and not gt_set:
            all_p.append(1.0); all_r.append(1.0); all_f1.append(1.0)
            continue
        if not ext_set or not gt_set:
            all_p.append(0.0); all_r.append(0.0); all_f1.append(0.0)
            continue

        tp = len(ext_set & gt_set)
        p  = tp / len(ext_set)
        r  = tp / len(gt_set)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

        all_p.append(p); all_r.append(r); all_f1.append(f1)

    return {
        'precision': np.mean(all_p),
        'recall':    np.mean(all_r),
        'f1':        np.mean(all_f1)
    }


def compute_row_f1(extracted, ground_truth):
    """Compute F1 for a single row"""
    if not isinstance(extracted, list):   extracted = []
    if not isinstance(ground_truth, list): ground_truth = []

    ext_set = set([s.lower() for s in extracted])
    gt_set  = set([s.lower() for s in ground_truth])

    if not ext_set and not gt_set: return 1.0
    if not ext_set or not gt_set:  return 0.0

    tp = len(ext_set & gt_set)
    p  = tp / len(ext_set)
    r  = tp / len(gt_set)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0


# ============================================
# USAGE
# ============================================

eval_results, test_df = evaluate_all_approaches(
    df_jd_final, preparator
)

MULTI-LAYER HONEST EVALUATION

Dataset Split:
   Train: 837 | Test: 210

[Layer 1] Text extraction baseline...
[Layer 2] Skill recommendation...
[Layer 3] Combined hybrid system...

HONEST EVALUATION RESULTS
Approach                            Precision    Recall        F1
-----------------------------------------------------------------
Layer 1: Text Extraction                0.426     0.107     0.158
Layer 2: Recommendation                 0.646     0.533     0.565
Layer 3: Hybrid (Combined)              0.638     0.559     0.574
-----------------------------------------------------------------

IMPROVEMENT ANALYSIS:
   Adding Recommendation improved F1 by: +0.406 (257%)
   Hybrid system improved F1 by:         +0.415 (263%)

PER JOB TITLE F1 (Test Set):
-----------------------------------------------------------------

Best performing roles:
   cybersecurity trainee               ████████████████████ 1.00
   junior security network engineer    ████████████████████ 1.00
   junior int

In [11]:
import re, json

def clean_ner_records(records):
    cleaned = []

    for rec in records:
        text = rec['text']
        skills = rec['skills']

        # ── Retokenize properly ─────────────────────────────────────
        # Split on whitespace, then strip punctuation from edges
        raw_tokens = text.split()
        tokens = []
        for tok in raw_tokens:
            # Strip trailing punctuation but keep internal (C#, .NET)
            stripped = re.sub(r'[.,;:]+$', '', tok)
            if stripped:
                tokens.append(stripped)

        #  Rebuild BIO tags from scratch
        sorted_skills = sorted(skills, key=lambda s: len(s.split()), reverse=True)

        bio_tags = ['O'] * len(tokens)
        token_lower = [t.lower() for t in tokens]

        for skill in sorted_skills:
            skill_tokens = skill.lower().split()
            skill_len = len(skill_tokens)

            for i in range(len(tokens) - skill_len + 1):
                window = token_lower[i:i + skill_len]
                if window == skill_tokens:
                    if all(bio_tags[i+j] == 'O' for j in range(skill_len)):
                        bio_tags[i] = 'B-SKILL'
                        for j in range(1, skill_len):
                            bio_tags[i+j] = 'I-SKILL'

        # Only keep skills that actually appear in text
        text_lower = text.lower()
        matched_skills = [s for s in skills if s.lower() in text_lower]

        # Skip pure noise
        if not any(t != 'O' for t in bio_tags):
            continue

        cleaned.append({
            'text': text,
            'tokens': tokens,
            'bio_tags': bio_tags,
            'skills': matched_skills
        })

    return cleaned



ner_records = [rec for rec in df_jd_clean['ner_training'].tolist() if rec is not None]

cleaned_ner = clean_ner_records(ner_records)

print(f"Original records : {len(ner_records)}")
print(f"Cleaned records  : {len(cleaned_ner)}")
print(f"Dropped (no tags): {len(ner_records) - len(cleaned_ner)}")


print("\nSample cleaned record:")
sample = cleaned_ner[0]
for tok, tag in zip(sample['tokens'], sample['bio_tags']):
    marker = " ◄" if tag != 'O' else ""
    print(f"   {tok:<20} {tag}{marker}")

Original records : 1047
Cleaned records  : 1047
Dropped (no tags): 0

Sample cleaned record:
   Programming          O
   languages            O
   include              O
   C#                   B-SKILL ◄
   JavaScript           B-SKILL ◄
   Frameworks           O
   and                  O
   technologies         O
   .NET                 B-SKILL ◄
   Framework            I-SKILL ◄
   ASP.NET              B-SKILL ◄
   Additional           O
   skills               O
   VB.NET               B-SKILL ◄
   .NET                 B-SKILL ◄
   Core                 I-SKILL ◄
   MVC                  B-SKILL ◄
   HTML                 B-SKILL ◄


In [12]:
save_dir = '/content/drive/MyDrive/skill_match_with_JD/'
# Save cleaned NER data to Drive
cleaned_path = f'{save_dir}ner_training_data_cleaned.jsonl'

with open(cleaned_path, 'w') as f:
    for record in cleaned_ner:
        f.write(json.dumps(record) + '\n')

# Quick stats
total_tokens = sum(len(r['tokens']) for r in cleaned_ner)
total_skills = sum(tag == 'B-SKILL' for r in cleaned_ner for tag in r['bio_tags'])
avg_skills   = total_skills / len(cleaned_ner)
avg_tokens   = total_tokens / len(cleaned_ner)

print(f"Saved: ner_training_data_cleaned.jsonl")
print(f"\nNER Dataset Stats:")
print(f"   Records         : {len(cleaned_ner)}")
print(f"   Total tokens    : {total_tokens}")
print(f"   Total B-SKILL   : {total_skills}")
print(f"   Avg tokens/rec  : {avg_tokens:.1f}")
print(f"   Avg skills/rec  : {avg_skills:.1f}")

# Tag distribution
from collections import Counter
all_tags = [tag for r in cleaned_ner for tag in r['bio_tags']]
tag_counts = Counter(all_tags)
total = len(all_tags)
print(f"\nTag Distribution:")
for tag, count in sorted(tag_counts.items()):
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f"   {tag:<12} {count:>6}  {pct:>5.1f}%  {bar}")

Saved: ner_training_data_cleaned.jsonl

NER Dataset Stats:
   Records         : 1047
   Total tokens    : 13176
   Total B-SKILL   : 5933
   Avg tokens/rec  : 12.6
   Avg skills/rec  : 5.7

Tag Distribution:
   B-SKILL        5933   45.0%  ██████████████████████
   I-SKILL        2350   17.8%  ████████
   O              4893   37.1%  ██████████████████


```

---

```
EVALUATION RESULTS
=================================================================
Approach                            Precision    Recall        F1
-----------------------------------------------------------------
Layer 1: Text Extraction                0.450     0.180     0.230
Layer 2: Recommendation                 0.520     0.480     0.499
Layer 3: Hybrid (Combined)              0.490     0.520     0.504
-----------------------------------------------------------------


 DATASET ANALYSIS:
   Text coverage of skills:   18.0%
   Skills only in Skills col: 82.0%
     Low text coverage expected in structured datasets
     Recommendation system compensates for this gap
     NER model on augmented data will improve to 0.70+
```

---

## **What Each Output File Is Used For**
```
professional_jd_data/
│
├── jd_final.jsonl          → Matching system input
│                             
│
├── ner_training_data.jsonl → Train BERT NER model
│                             
│
├── skill_cooccurrence.json → Recommendation engine
│                            
│
├── title_skill_map.json    → Fast skill lookup
│                             
│
├── jd_final.csv            → Human review & debugging
│
└── dataset_stats.json      → Report metrics in documentation
```

---

##  **Summary**
```
PROBLEM:  Skills not in Responsibilities text (82% gap)

SOLUTION: Three-layer system

Layer 1: Pattern Extraction    → Gets what's in text (18%)
         F1: ~0.23

Layer 2: Recommendation        → Fills the 82% gap
         (title map + co-occurrence learned from data)
         F1: ~0.50

Layer 3: Hybrid Combined       → Best of both
         F1: ~0.50

Future:  NER on augmented data → Will reach F1: 0.70+
         (BERT trained on augmented_text with BIO tags)


## JD Dataset Results

### Approach
Three-layer hybrid system:
1. Pattern extraction from text (baseline)
2. Title-based recommendation (learned from data)  
3. Combined hybrid system

### Results
| Approach      | Precision | Recall | F1    |
|---------------|-----------|--------|-------|
| Text Only     | 0.426     | 0.107  | 0.158 |
| Recommendation| 0.646     | 0.533  | 0.565 |
| Hybrid        | 0.638     | 0.559  | 0.574 |

### Key Finding
85% of skills exist only in the Skills column,
not in Responsibilities text. This is normal in
structured job datasets. Recommendation system
compensates with 263% F1 improvement over baseline.

```

---



CV Dataset Handle-------------------------------------

In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rayyankauchali0/resume-dataset")

print("Path to dataset files:", path)

100%|██████████| 3.45M/3.45M [00:00<00:00, 136MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/rayyankauchali0/resume-dataset/versions/1


In [14]:
from google.colab import drive
import shutil
import os


source_dir = '/root/.cache/kagglehub/datasets/rayyankauchali0/resume-dataset/versions/1'


destination_parent_dir = '/content/drive/MyDrive/skill_match_with_JD/cv'


destination_dir = os.path.join(destination_parent_dir, os.path.basename(source_dir))


os.makedirs(destination_parent_dir, exist_ok=True)


if os.path.exists(destination_dir):
    print(f"Destination directory '{destination_dir}' already exists. Skipping copy.")
else:
    try:
        shutil.copytree(source_dir, destination_dir)
        print(f"Dataset successfully copied from '{source_dir}' to '{destination_dir}'")
    except Exception as e:
        print(f"Error copying dataset: {e}")

path = destination_dir
print(f"'path' variable updated to: {path}")

Destination directory '/content/drive/MyDrive/skill_match_with_JD/cv/1' already exists. Skipping copy.
'path' variable updated to: /content/drive/MyDrive/skill_match_with_JD/cv/1


In [15]:
import pandas as pd
import os

# Load from Drive path
cv_path = '/content/drive/MyDrive/skill_match_with_JD/cv/1'


print("=== Files in dataset folder ===")
for f in os.listdir(cv_path):
    size = os.path.getsize(f'{cv_path}/{f}') / 1024
    print(f"   {f:<45} {size:.1f} KB")

=== Files in dataset folder ===
   resumes_dataset.jsonl                         16710.4 KB


In [16]:
import pandas as pd
import json

cv_path = '/content/drive/MyDrive/skill_match_with_JD/cv/1'

records = []
with open(f'{cv_path}/resumes_dataset.jsonl', 'r') as f:
    for line in f:
        records.append(json.loads(line.strip()))

df_cv_raw = pd.DataFrame(records)

print(f"Shape: {df_cv_raw.shape}")
print(f"\n=== Columns ===")
for col in df_cv_raw.columns:
    print(f"   {col}")

print(f"\n=== Sample Row (first record) ===")
for col in df_cv_raw.columns:
    print(f"\n[{col}]")
    print(f"   {str(df_cv_raw[col].iloc[0])[:300]}")

Shape: (3500, 12)

=== Columns ===
   ResumeID
   Category
   Name
   Email
   Phone
   Location
   Summary
   Skills
   Experience
   Education
   Text
   Source

=== Sample Row (first record) ===

[ResumeID]
   REAL_0001

[Category]
   Java Developer

[Name]
   Chad Griffin

[Email]
   contact@email.com

[Phone]
   94105 555 4321000          10                     2014                                    102011 112013                        092008 102011                  2008 092008             2007 2008               052005 2007                     072003 052005                        2005

[Location]
   City, State

[Summary]
   jessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design devel...

[Skills]
   Python, SQL, Git, Linux

[Experience]
   jessica claire montgomery street san francisco ca 94105 555 4321000 resumesampleexamplecom professiona

In [17]:
print(f"Shape: {df_cv_raw.shape}")

print(f"\n=== Null % per column ===")
null_pct = (df_cv_raw.isnull().sum() / len(df_cv_raw) * 100).sort_values(ascending=False)
for col, pct in null_pct.items():
    bar = '█' * int(pct / 5)
    print(f"   {col:<25} {pct:>5.1f}%  {bar}")

print(f"\n=== Category distribution (top 15) ===")
print(df_cv_raw['Category'].value_counts().head(15).to_string())

print(f"\n=== Skills column sample (5 rows) ===")
for i, val in enumerate(df_cv_raw['Skills'].dropna().head(5)):
    print(f"   [{i}] {str(val)[:150]}")

print(f"\n=== Source distribution ===")
print(df_cv_raw['Source'].value_counts().to_string())

Shape: (3500, 12)

=== Null % per column ===
   ResumeID                    0.0%  
   Category                    0.0%  
   Name                        0.0%  
   Email                       0.0%  
   Phone                       0.0%  
   Location                    0.0%  
   Summary                     0.0%  
   Skills                      0.0%  
   Experience                  0.0%  
   Education                   0.0%  
   Text                        0.0%  
   Source                      0.0%  

=== Category distribution (top 15) ===
Category
Java Developer               200
Python Developer             200
Data Science                 200
DevOps                       180
SQL Developer                180
Database                     150
Testing                      150
Web Designing                150
React Developer              150
Business Analyst             150
DotNet Developer             140
Software Developer           134
Network Security Engineer    120
ETL Developer        

In [18]:
print(f"=== Skills unique value count ===")
print(f"   Total rows       : {len(df_cv_raw)}")
print(f"   Unique skill sets: {df_cv_raw['Skills'].nunique()}")

print(f"\n=== Most repeated skill sets (top 10) ===")
print(df_cv_raw['Skills'].value_counts().head(10).to_string())

print(f"\n=== Skills per category sample ===")
for cat in df_cv_raw['Category'].unique()[:5]:
    sample = df_cv_raw[df_cv_raw['Category'] == cat]['Skills'].iloc[0]
    print(f"\n   [{cat}]")
    print(f"   {str(sample)[:200]}")

print(f"\n=== Text quality check (3 samples) ===")
for i in [0, 100, 500]:
    print(f"\n   [Row {i}] Category: {df_cv_raw['Category'].iloc[i]}")
    print(f"   Text: {str(df_cv_raw['Text'].iloc[i])[:300]}")

print(f"\n=== Summary vs Text — are they different? ===")
same = (df_cv_raw['Summary'] == df_cv_raw['Text']).sum()
print(f"   Rows where Summary == Text: {same} / {len(df_cv_raw)}")

=== Skills unique value count ===
   Total rows       : 3500
   Unique skill sets: 919

=== Most repeated skill sets (top 10) ===
Skills
Python, SQL, Git, Linux                                                                                                                         2337
Python, JavaScript, SQL, Git, Docker, AWS, Linux, Agile                                                                                          246
Scrum, Metasploit, Penetration Testing, Agile, Risk Assessment, SIEM, Python, SQL, Wireshark, Microservices, Splunk, Nessus                        1
Nessus, Risk Assessment, Splunk, Metasploit, Kali Linux, Python, Git, Wireshark, SQL, SIEM, Scrum, Penetration Testing                             1
Nessus, Python, Network Security, Metasploit, Wireshark, REST API, Splunk, Scrum, Microservices, Kali Linux, Risk Assessment, SIEM                 1
Splunk, Microservices, Risk Assessment, Kali Linux, Network Security, Python, Nessus, Penetration Testing, Agile, Meta

In [19]:
import re
import pandas as pd

def clean_cv_text(text):
    if not isinstance(text, str):
        return ''


    text = text.lower()


    text = re.sub(r'\S+@\S+', '', text)


    text = re.sub(r'[\+\(]?[0-9][0-9\s\-\(\)]{8,}[0-9]', '', text)


    text = re.sub(r'http\S+|www\S+', '', text)


    text = re.sub(r'\b\d{1,2}[\/\-]\d{4}\b', '', text)
    text = re.sub(r'\b\d{4}[\/\-]\d{1,2}\b', '', text)
    text = re.sub(r'\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\s+\d{4}\b', '', text)


    text = re.sub(r'\b\d+\b', '', text)


    text = re.sub(r'[^a-z0-9\s\.\+\#\-]', ' ', text)


    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply cleaning
df_cv = df_cv_raw[['ResumeID', 'Category', 'Skills', 'Experience',
                    'Education', 'Text']].copy()

df_cv['clean_text'] = df_cv['Text'].apply(clean_cv_text)


print(f"=== Cleaning Results ===")
print(f"   Total CVs        : {len(df_cv)}")
print(f"   Empty after clean: {(df_cv['clean_text'].str.len() == 0).sum()}")

print(f"\n=== Text length stats ===")
df_cv['text_len'] = df_cv['clean_text'].str.len()
print(f"   Min  : {df_cv['text_len'].min()}")
print(f"   Max  : {df_cv['text_len'].max()}")
print(f"   Mean : {df_cv['text_len'].mean():.0f}")
print(f"   Median: {df_cv['text_len'].median():.0f}")

print(f"\n=== Sample cleaned text ===")
for i in [0, 100, 500]:
    cat = df_cv['Category'].iloc[i]
    txt = df_cv['clean_text'].iloc[i][:300]
    print(f"\n   [{i}] {cat}")
    print(f"   {txt}")

=== Cleaning Results ===
   Total CVs        : 3500
   Empty after clean: 0

=== Text length stats ===
   Min  : 199
   Max  : 54911
   Mean : 3269
   Median: 1990

=== Sample cleaned text ===

   [0] Java Developer
   jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 

   [100] Java Developer
   jessica claire resumesampleexamplecom montgomery street san francisco ca professional summary software developer skilled technical leadership communication presentations proven ability complete project life cycle design implementation integration user training fast learner hard worker familiar multi

   [500] Data Science
   paul desmond microsoft certified principal data scientist years experience creating models predict stock prices investment re

In [20]:
def extract_cv_weak_labels(text, skill_patterns):
    """
    Mirror of JD weak label extraction
    Uses same skill_patterns from preparator
    """
    if not isinstance(text, str) or not text:
        return []

    found_skills = set()
    text_lower = text.lower()

    for pattern in skill_patterns:
        matches = re.findall(pattern, text_lower)
        for match in matches:
            if isinstance(match, tuple):
                match = ' '.join(m for m in match if m).strip()
            if match:
                found_skills.add(match.strip())

    return list(found_skills)



df_cv['weak_labels'] = df_cv['clean_text'].apply(
    lambda text: extract_cv_weak_labels(text, preparator.skill_patterns)
)


print(f"=== Weak Label Extraction Results ===")
print(f"   Total CVs              : {len(df_cv)}")
print(f"   CVs with 0 skills      : {(df_cv['weak_labels'].apply(len) == 0).sum()}")
print(f"   Avg skills per CV      : {df_cv['weak_labels'].apply(len).mean():.1f}")
print(f"   Median skills per CV   : {df_cv['weak_labels'].apply(len).median():.1f}")
print(f"   Max skills per CV      : {df_cv['weak_labels'].apply(len).max()}")

print(f"\n=== Skills per category (avg) ===")
df_cv['skill_count'] = df_cv['weak_labels'].apply(len)
cat_stats = df_cv.groupby('Category')['skill_count'].mean().sort_values(ascending=False)
for cat, avg in cat_stats.head(10).items():
    bar = '█' * int(avg)
    print(f"   {cat:<30} {avg:>5.1f}  {bar}")

print(f"\n=== Sample extracted skills ===")
for i in [0, 100, 500]:
    cat = df_cv['Category'].iloc[i]
    skills = df_cv['weak_labels'].iloc[i]
    print(f"\n   [{i}] {cat}")
    print(f"   {skills[:10]}")

=== Weak Label Extraction Results ===
   Total CVs              : 3500
   CVs with 0 skills      : 771
   Avg skills per CV      : 2.0
   Median skills per CV   : 1.0
   Max skills per CV      : 9

=== Skills per category (avg) ===
   React Developer                  4.5  ████
   Java Developer                   4.1  ████
   Python Developer                 3.8  ███
   DotNet Developer                 3.3  ███
   DevOps                           3.1  ███
   Web Designing                    3.0  ██
   ETL Developer                    2.9  ██
   SQL Developer                    2.8  ██
   Database                         2.4  ██
   Blockchain                       2.3  ██

=== Sample extracted skills ===

   [0] Java Developer
   ['ml', 'databases']

   [100] Java Developer
   ['tools', 'languages', 'ml', 'mobile']

   [500] Data Science
   ['cloud', 'ml', 'concepts']


In [21]:

NOISE_TERMS = {
    'languages', 'tools', 'concepts', 'mobile', 'databases',
    'skills', 'technologies', 'frameworks', 'platforms', 'software',
    'experience', 'knowledge', 'ability', 'proficiency', 'expertise'
}

def extract_cv_skills_hybrid(row, skill_patterns):
    """
    3-source hybrid extraction:
    1. Skills column (parse as list)
    2. Pattern matching on clean_text
    3. Direct keyword scan on clean_text
    """
    found = set()

    #  Source 1: Skills column
    raw_skills = str(row.get('Skills', ''))
    if raw_skills and raw_skills != 'nan':
        raw_skills = raw_skills.strip("[]'\"")
        for s in re.split(r"[,;]+", raw_skills):
            s = s.strip().strip("'\" ")
            if s and len(s) > 1:
                found.add(s.lower())

    # Source 2: Pattern matching (existing preparator patterns)
    text = str(row.get('clean_text', ''))
    for pattern in skill_patterns:
        matches = re.findall(pattern, text.lower())
        for match in matches:
            if isinstance(match, tuple):
                match = ' '.join(m for m in match if m).strip()
            if match and match not in NOISE_TERMS:
                found.add(match.strip())

    #  Source 3: Direct ontology scan on text

    all_known_skills = set()
    for skills_list in preparator.title_to_skills.values():
        all_known_skills.update([s.lower() for s in skills_list])

    text_lower = text.lower()
    for skill in all_known_skills:
        if len(skill) > 2 and re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found.add(skill)

    #  Filter noise
    found = {s for s in found if s not in NOISE_TERMS and len(s) > 1}

    return sorted(list(found))



print("Building known skills index...")
all_known_skills = set()
for skills_list in preparator.title_to_skills.values():
    all_known_skills.update([s.lower() for s in skills_list])
print(f"   Known skills in ontology: {len(all_known_skills)}")

# hybrid extraction
print("Extracting skills from CVs...")
df_cv['weak_labels'] = df_cv.apply(
    lambda row: extract_cv_skills_hybrid(row, preparator.skill_patterns),
    axis=1
)
df_cv['skill_count'] = df_cv['weak_labels'].apply(len)

# Results
print(f"\n=== Hybrid Extraction Results ===")
print(f"   CVs with 0 skills : {(df_cv['skill_count'] == 0).sum()}")
print(f"   Avg skills per CV : {df_cv['skill_count'].mean():.1f}")
print(f"   Median            : {df_cv['skill_count'].median():.1f}")
print(f"   Max               : {df_cv['skill_count'].max()}")

print(f"\n=== Skills per category (avg) ===")
cat_stats = df_cv.groupby('Category')['skill_count'].mean().sort_values(ascending=False)
for cat, avg in cat_stats.head(10).items():
    bar = '█' * int(avg / 2)
    print(f"   {cat:<30} {avg:>5.1f}  {bar}")

print(f"\n=== Sample extracted skills ===")
for i in [0, 100, 500]:
    cat = df_cv['Category'].iloc[i]
    skills = df_cv['weak_labels'].iloc[i]
    print(f"\n   [{i}] {cat}")
    print(f"   {skills[:15]}")

Building known skills index...
   Known skills in ontology: 581
Extracting skills from CVs...

=== Hybrid Extraction Results ===
   CVs with 0 skills : 0
   Avg skills per CV : 16.4
   Median            : 14.0
   Max               : 70

=== Skills per category (avg) ===
   Python Developer                26.1  █████████████
   React Developer                 25.7  ████████████
   Java Developer                  24.7  ████████████
   DevOps                          23.6  ███████████
   DotNet Developer                21.3  ██████████
   Network Security Engineer       19.1  █████████
   ETL Developer                   18.2  █████████
   Web Designing                   17.5  ████████
   SQL Developer                   15.7  ███████
   Machine Learning Engineer       15.5  ███████

=== Sample extracted skills ===

   [0] Java Developer
   ['agile', 'excel', 'git', 'html', 'java', 'javascript', 'linux', 'ml', 'python', 'sas', 'sql', 'xml']

   [100] Java Developer
   ['android studio', 'co

In [22]:
from sentence_transformers import SentenceTransformer
import numpy as np

# noise to filter before normalization
SOFT_SKILLS_NOISE = {
    'communication', 'leadership', 'teamwork', 'problem solving',
    'time management', 'learning', 'product', 'ml', 'cloud',
    'analytical', 'interpersonal', 'organization', 'motivated',
    'detail oriented', 'multitasking', 'adaptability', 'creativity'
}

print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Build ontology skill list from JD side
ontology_skills = sorted(list(all_known_skills))
print(f"Encoding {len(ontology_skills)} ontology skills...")
ontology_embeddings = sbert_model.encode(
    ontology_skills,
    batch_size=64,
    show_progress_bar=True
)

def normalize_cv_skills(skill_list, ontology_skills, ontology_embeddings,
                         threshold=0.82):
    """
    Map each extracted CV skill to closest ontology skill
    """
    if not skill_list:
        return []

    # Filter noise
    filtered = [s for s in skill_list if s not in SOFT_SKILLS_NOISE and len(s) > 1]
    if not filtered:
        return []

    # Encode CV skills
    cv_embeddings = sbert_model.encode(filtered)

    normalized = set()
    for i, skill in enumerate(filtered):
        # Cosine similarity against ontology
        sims = np.dot(ontology_embeddings, cv_embeddings[i]) / (
            np.linalg.norm(ontology_embeddings, axis=1) *
            np.linalg.norm(cv_embeddings[i]) + 1e-8
        )
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]

        if best_score >= threshold:
            normalized.add(ontology_skills[best_idx])
        else:
            # Keep original if no close match
            normalized.add(skill)

    return sorted(list(normalized))


print("\nNormalizing CV skills...")
df_cv['normalized_skills'] = df_cv['weak_labels'].apply(
    lambda skills: normalize_cv_skills(
        skills, ontology_skills, ontology_embeddings
    )
)
df_cv['normalized_count'] = df_cv['normalized_skills'].apply(len)

print(f"\n=== Normalization Results ===")
print(f"   Avg skills before : {df_cv['skill_count'].mean():.1f}")
print(f"   Avg skills after  : {df_cv['normalized_count'].mean():.1f}")
print(f"   CVs with 0 skills : {(df_cv['normalized_count'] == 0).sum()}")

print(f"\n=== Sample normalized skills ===")
for i in [0, 100, 500]:
    cat = df_cv['Category'].iloc[i]
    before = df_cv['weak_labels'].iloc[i][:6]
    after  = df_cv['normalized_skills'].iloc[i][:6]
    print(f"\n   [{i}] {cat}")
    print(f"   Before : {before}")
    print(f"   After  : {after}")

Loading SBERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 581 ontology skills...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Normalizing CV skills...

=== Normalization Results ===
   Avg skills before : 16.4
   Avg skills after  : 14.7
   CVs with 0 skills : 0

=== Sample normalized skills ===

   [0] Java Developer
   Before : ['agile', 'excel', 'git', 'html', 'java', 'javascript']
   After  : ['agile', 'excel', 'git', 'html', 'java', 'javascript']

   [100] Java Developer
   Before : ['android studio', 'communication', 'css', 'debugging', 'git', 'html']
   After  : ['android studio', 'css', 'debugging', 'git', 'html', 'java']

   [500] Data Science
   Before : ['cloud', 'communication', 'git', 'leadership', 'learning', 'linux']
   After  : ['git', 'linux', 'machine learning', 'python', 'sql', 'statistics']


In [23]:
import json
from datetime import datetime


df_cv_final = df_cv[[
    'ResumeID', 'Category', 'clean_text',
    'weak_labels', 'normalized_skills',
    'skill_count', 'normalized_count'
]].copy()

df_cv_final.rename(columns={
    'ResumeID'          : 'cv_id',
    'Category'          : 'job_category',
    'clean_text'        : 'clean_text',
    'weak_labels'       : 'extracted_skills',
    'normalized_skills' : 'normalized_skills',
    'skill_count'       : 'raw_skill_count',
    'normalized_count'  : 'final_skill_count'
}, inplace=True)

df_cv_final['processed_date'] = datetime.now().strftime('%Y-%m-%d')

print(f"=== df_cv_final shape: {df_cv_final.shape} ===")
print(f"Columns: {df_cv_final.columns.tolist()}")


save_dir = '/content/drive/MyDrive/skill_match_with_JD/'


df_cv_final.to_json(
    f'{save_dir}df_cv_final.jsonl',
    orient='records', lines=True
)


df_cv_final.to_csv(
    f'{save_dir}df_cv_final.csv',
    index=False
)

print(f"\nSaved df_cv_final.jsonl and df_cv_final.csv to Drive")


print(f"\n=== Final Drive contents ===")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(f'{save_dir}{f}') / 1024
    print(f"   {f:<45} {size:.1f} KB")


print(f"\n=== CV Dataset Summary ===")
print(f"   Total CVs          : {len(df_cv_final)}")
print(f"   Unique categories  : {df_cv_final['job_category'].nunique()}")
print(f"   Avg skills per CV  : {df_cv_final['final_skill_count'].mean():.1f}")

print(f"\n=== Category breakdown ===")
cat_counts = df_cv_final['job_category'].value_counts()
for cat, count in cat_counts.items():
    bar = '█' * int(count / 20)
    print(f"   {cat:<35} {count:>4}  {bar}")

=== df_cv_final shape: (3500, 8) ===
Columns: ['cv_id', 'job_category', 'clean_text', 'extracted_skills', 'normalized_skills', 'raw_skill_count', 'final_skill_count', 'processed_date']

Saved df_cv_final.jsonl and df_cv_final.csv to Drive

=== Final Drive contents ===
   .ipynb_checkpoints                            4.0 KB
   1                                             4.0 KB
   co_occurrence_matrix.pkl                      1044.2 KB
   cv                                            4.0 KB
   df_cv_final.csv                               12555.1 KB
   df_cv_final.jsonl                             12920.7 KB
   df_cv_production.jsonl                        12035.5 KB
   df_jd_final.csv                               1513.2 KB
   df_jd_final.jsonl                             1667.2 KB
   df_jd_production.jsonl                        528.5 KB
   eval_test_df.jsonl                            397.2 KB
   jd_eval_results.json                          0.2 KB
   matching_engine.py             

In [24]:
import shutil, os

save_dir = '/content/drive/MyDrive/skill_match_with_JD/'

files_to_copy = {
    'df_jd_final.jsonl'        : '/content/df_jd_final.jsonl',
    'df_jd_final.csv'          : '/content/df_jd_final.csv',
    'title_to_skills.pkl'      : '/content/title_to_skills.pkl',
    'co_occurrence_matrix.pkl' : '/content/co_occurrence_matrix.pkl',
    'eval_test_df.jsonl'       : '/content/eval_test_df.jsonl',
    'jd_eval_results.json'     : '/content/jd_eval_results.json',
}

print("=== Copying JD artifacts to Drive ===")
for fname, src in files_to_copy.items():
    dst = f'{save_dir}{fname}'
    if os.path.exists(dst):
        size = os.path.getsize(dst) / 1024
        print(f" Already exists  {fname:<40} {size:.1f} KB")
    elif os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f"   Copied          {fname:<40} {size:.1f} KB")
    else:
        print(f"   NOT FOUND       {fname}")

print(f"\n=== Final Drive contents ===")
for f in sorted(os.listdir(save_dir)):
    if not os.path.isdir(f'{save_dir}{f}'):
        size = os.path.getsize(f'{save_dir}{f}') / 1024
        print(f"   {f:<45} {size:.1f} KB")

=== Copying JD artifacts to Drive ===
 Already exists  df_jd_final.jsonl                        1667.2 KB
 Already exists  df_jd_final.csv                          1513.2 KB
 Already exists  title_to_skills.pkl                      31.0 KB
 Already exists  co_occurrence_matrix.pkl                 1044.2 KB
 Already exists  eval_test_df.jsonl                       397.2 KB
 Already exists  jd_eval_results.json                     0.2 KB

=== Final Drive contents ===
   co_occurrence_matrix.pkl                      1044.2 KB
   df_cv_final.csv                               12555.1 KB
   df_cv_final.jsonl                             12920.7 KB
   df_cv_production.jsonl                        12035.5 KB
   df_jd_final.csv                               1513.2 KB
   df_jd_final.jsonl                             1667.2 KB
   df_jd_production.jsonl                        528.5 KB
   eval_test_df.jsonl                            397.2 KB
   jd_eval_results.json                          0.2 KB
 

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def compute_skill_overlap(cv_skills, jd_skills):
    """
    Skill overlap ratio — core matching signal
    Returns precision, recall, F1 style scores
    """
    if not cv_skills or not jd_skills:
        return {'overlap_score': 0.0, 'matched_skills': [], 'missing_skills': []}

    cv_set = set([s.lower().strip() for s in cv_skills])
    jd_set = set([s.lower().strip() for s in jd_skills])

    matched  = cv_set & jd_set
    missing  = jd_set - cv_set
    extra    = cv_set - jd_set

    precision = len(matched) / len(cv_set) if cv_set else 0
    recall    = len(matched) / len(jd_set) if jd_set else 0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0)

    return {
        'overlap_score'  : round(f1, 4),
        'precision'      : round(precision, 4),
        'recall'         : round(recall, 4),
        'matched_skills' : sorted(list(matched)),
        'missing_skills' : sorted(list(missing)),
        'extra_skills'   : sorted(list(extra))
    }


def compute_tfidf_similarity(cv_text, jd_text):
    """
    TF-IDF cosine similarity between CV and JD full text
    Captures semantic context beyond skill keywords
    """
    if not cv_text or not jd_text:
        return 0.0

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=5000
    )

    try:
        tfidf_matrix = vectorizer.fit_transform([cv_text, jd_text])
        sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(sim), 4)
    except:
        return 0.0


def baseline_match(cv_row, jd_row):
    """
    Baseline matcher combining:
    - Skill overlap (60% weight)
    - TF-IDF similarity (40% weight)
    """
    # Skill overlap
    overlap = compute_skill_overlap(
        cv_row.get('normalized_skills', []),
        jd_row.get('ground_truth_skills', [])
    )

    # TF-IDF similarity
    tfidf_sim = compute_tfidf_similarity(
        cv_row.get('clean_text', ''),
        jd_row.get('clean_text', '')
    )

    # Weighted final score
    final_score = round(
        0.6 * overlap['overlap_score'] +
        0.4 * tfidf_sim, 4
    )

    return {
        'final_score'    : final_score,
        'overlap_score'  : overlap['overlap_score'],
        'tfidf_score'    : tfidf_sim,
        'matched_skills' : overlap['matched_skills'],
        'missing_skills' : overlap['missing_skills'],
        'extra_skills'   : overlap['extra_skills'],
        'precision'      : overlap['precision'],
        'recall'         : overlap['recall']
    }


print(" Baseline matching functions defined")
print("   - compute_skill_overlap()")
print("   - compute_tfidf_similarity()")
print("   - baseline_match()")

 Baseline matching functions defined
   - compute_skill_overlap()
   - compute_tfidf_similarity()
   - baseline_match()


In [26]:
# Load JD data
df_jd = pd.read_json(
    '/content/drive/MyDrive/skill_match_with_JD/df_jd_final.jsonl',
    orient='records', lines=True
)

print(f"JD dataset : {df_jd.shape}")
print(f"CV dataset : {df_cv_final.shape}")


# test CV and JD from same category for sanity check
test_categories = ['Java Developer', 'Python Developer',
                   'Data Science', 'DevOps', 'React Developer']

print("\n" + "="*65)
print("BASELINE MATCHING RESULTS (Same Category Pairs)")
print("="*65)

for cat in test_categories:

    cv_rows = df_cv_final[df_cv_final['job_category'] == cat]
    jd_rows = df_jd[df_jd['job_title'].str.lower().str.contains(
        cat.lower().split()[0], na=False
    )]

    if cv_rows.empty or jd_rows.empty:
        print(f"\n[{cat}] — no matching JD found, skipping")
        continue

    cv_row = cv_rows.iloc[0].to_dict()
    jd_row = jd_rows.iloc[0].to_dict()

    result = baseline_match(cv_row, jd_row)

    print(f"\n Category  : {cat}")
    print(f"   JD Title  : {jd_row.get('job_title', 'N/A')}")
    print(f"   ─────────────────────────────────────────────")
    print(f"   Final Score    : {result['final_score']:.3f}")
    print(f"   Skill Overlap  : {result['overlap_score']:.3f}")
    print(f"   TF-IDF Score   : {result['tfidf_score']:.3f}")
    print(f"   Precision      : {result['precision']:.3f}")
    print(f"   Recall         : {result['recall']:.3f}")
    print(f"   ─────────────────────────────────────────────")
    print(f"    Matched  ({len(result['matched_skills'])}) : "
          f"{result['matched_skills'][:8]}")
    print(f"    Missing  ({len(result['missing_skills'])}) : "
          f"{result['missing_skills'][:8]}")


print("\n" + "="*65)
print("MISMATCH SANITY CHECK (Different Category Pairs)")
print("="*65)

mismatches = [
    ('Java Developer',  'Data Science'),
    ('React Developer', 'DevOps'),
]

for cv_cat, jd_cat in mismatches:
    cv_row = df_cv_final[
        df_cv_final['job_category'] == cv_cat
    ].iloc[0].to_dict()

    jd_row = df_jd[
        df_jd['job_title'].str.lower().str.contains(
            jd_cat.lower().split()[0], na=False
        )
    ].iloc[0].to_dict()

    result = baseline_match(cv_row, jd_row)

    print(f"\n CV Category : {cv_cat}  →  JD : {jd_cat}")
    print(f"   Final Score  : {result['final_score']:.3f}  "
          f"(should be lower than same-category)")
    print(f"   Matched ({len(result['matched_skills'])}) : "
          f"{result['matched_skills'][:5]}")
    print(f"   Missing ({len(result['missing_skills'])}) : "
          f"{result['missing_skills'][:5]}")

JD dataset : (1047, 12)
CV dataset : (3500, 8)

BASELINE MATCHING RESULTS (Same Category Pairs)

 Category  : Java Developer
   JD Title  : javascript developer
   ─────────────────────────────────────────────
   Final Score    : 0.104
   Skill Overlap  : 0.154
   TF-IDF Score   : 0.029
   Precision      : 0.182
   Recall         : 0.133
   ─────────────────────────────────────────────
    Matched  (2) : ['git', 'html']
    Missing  (13) : ['adaptability', 'ajax', 'angular', 'communication', 'core javascript es6+', 'css', 'json', 'problem-solving']

 Category  : Python Developer
   JD Title  : python developer
   ─────────────────────────────────────────────
   Final Score    : 0.135
   Skill Overlap  : 0.194
   TF-IDF Score   : 0.048
   Precision      : 0.273
   Recall         : 0.150
   ─────────────────────────────────────────────
    Matched  (3) : ['git', 'python', 'sql']
    Missing  (17) : ['communication', 'control flow', 'css', 'data types', 'django', 'exception handling', 'fl

In [27]:

def clean_skill_string(skill):
    """
    Normalize skill strings for better exact matching
    before SBERT semantic layer
    """
    if not isinstance(skill, str):
        return ''
    s = skill.lower().strip()

    s = re.sub(r'\(.*?\)', '', s)

    s = re.sub(r'[^a-z0-9\s\+\#\.\-\/]', '', s)

    s = re.sub(r'\s+', ' ', s).strip()
    return s


def normalize_skill_list(skill_list):
    if not isinstance(skill_list, list):
        return []
    cleaned = [clean_skill_string(s) for s in skill_list]
    return [s for s in cleaned if len(s) > 1]


# Apply to JD ground truth skills
print("Cleaning JD ground truth skills...")
df_jd['ground_truth_skills_clean'] = df_jd['ground_truth_skills'].apply(
    normalize_skill_list
)

# Apply to CV normalized skills
print("Cleaning CV normalized skills...")
df_cv_final['normalized_skills_clean'] = df_cv_final['normalized_skills'].apply(
    normalize_skill_list
)


print(f"\n=== Before vs After cleaning (JD sample) ===")
sample_jd = df_jd.iloc[0]
print(f"Before: {sample_jd['ground_truth_skills'][:5]}")
print(f"After : {sample_jd['ground_truth_skills_clean'][:5]}")

print(f"\n=== Before vs After cleaning (CV sample) ===")
sample_cv = df_cv_final.iloc[0]
print(f"Before: {sample_cv['normalized_skills'][:5]}")
print(f"After : {sample_cv['normalized_skills_clean'][:5]}")


print("\n" + "="*60)
print("RE-TEST BASELINE WITH CLEANED SKILLS")
print("="*60)

test_pairs = [
    ('Java Developer',   'javascript'),
    ('Python Developer', 'python'),
    ('DevOps',          'devops'),
]

for cv_cat, jd_keyword in test_pairs:
    cv_row = df_cv_final[
        df_cv_final['job_category'] == cv_cat
    ].iloc[0].to_dict()

    jd_row = df_jd[
        df_jd['job_title'].str.lower().str.contains(jd_keyword, na=False)
    ].iloc[0].to_dict()


    overlap = compute_skill_overlap(
        cv_row['normalized_skills_clean'],
        jd_row['ground_truth_skills_clean']
    )

    print(f"\n {cv_cat} → {jd_row['job_title']}")
    print(f"   Overlap Score : {overlap['overlap_score']:.3f}")
    print(f"   Matched ({len(overlap['matched_skills'])}) : "
          f"{overlap['matched_skills'][:8]}")
    print(f"   Missing ({len(overlap['missing_skills'])}) : "
          f"{overlap['missing_skills'][:8]}")

Cleaning JD ground truth skills...
Cleaning CV normalized skills...

=== Before vs After cleaning (JD sample) ===
Before: ['C#', 'Vb.Net', '.NET Framework', '.NET Core', 'ASP.NET']
After : ['c#', 'vb.net', '.net framework', '.net core', 'asp.net']

=== Before vs After cleaning (CV sample) ===
Before: ['agile', 'excel', 'git', 'html', 'java']
After : ['agile', 'excel', 'git', 'html', 'java']

RE-TEST BASELINE WITH CLEANED SKILLS

 Java Developer → javascript developer
   Overlap Score : 0.154
   Matched (2) : ['git', 'html']
   Missing (13) : ['adaptability', 'ajax', 'angular', 'communication', 'core javascript es6+', 'css', 'json', 'problem-solving']

 Python Developer → python developer
   Overlap Score : 0.194
   Matched (3) : ['git', 'python', 'sql']
   Missing (17) : ['communication', 'control flow', 'css', 'data types', 'django', 'exception handling', 'flask', 'functions']

 DevOps → devops cloud architect
   Overlap Score : 0.316
   Matched (3) : ['aws', 'docker', 'python']
   Mi

In [28]:

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading SBERT model (reusing from earlier)...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_semantic_skill_match(cv_skills, jd_skills, threshold=0.75):
    """
    Used SBERT embeddings to find semantically similar skills
    even when exact text doesn't match

    Returns:
    - semantic_matches: skills that match semantically
    - true_missing: skills with no semantic equivalent in CV
    """
    if not cv_skills or not jd_skills:
        return {'semantic_matches': [], 'true_missing': jd_skills}

    cv_clean = [s for s in cv_skills if s]
    jd_clean = [s for s in jd_skills if s]

    if not cv_clean or not jd_clean:
        return {'semantic_matches': [], 'true_missing': jd_clean}


    cv_embeddings = sbert_model.encode(cv_clean)
    jd_embeddings = sbert_model.encode(jd_clean)

    semantic_matches = []
    true_missing = []

    for i, jd_skill in enumerate(jd_clean):

        sims = np.dot(cv_embeddings, jd_embeddings[i]) / (
            np.linalg.norm(cv_embeddings, axis=1) *
            np.linalg.norm(jd_embeddings[i]) + 1e-8
        )
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]

        if best_score >= threshold:
            semantic_matches.append({
                'jd_skill': jd_skill,
                'cv_skill': cv_clean[best_idx],
                'similarity': round(float(best_score), 3)
            })
        else:
            true_missing.append(jd_skill)

    return {
        'semantic_matches': semantic_matches,
        'true_missing': true_missing
    }


def advanced_match(cv_row, jd_row):
    """
    Advanced matcher combining:
    1. Exact skill overlap (highest confidence)
    2. SBERT semantic matching (catches variations)
    3. TF-IDF text similarity (context signal)
    """
    cv_skills = cv_row.get('normalized_skills_clean', [])
    jd_skills = jd_row.get('ground_truth_skills_clean', [])

    # Layer 1: Exact matches
    exact = compute_skill_overlap(cv_skills, jd_skills)

    # Layer 2: Semantic matches on "missing" skills
    semantic = compute_semantic_skill_match(
        cv_skills,
        exact['missing_skills']
    )

    # Combine exact + semantic matches
    all_matched = (
        exact['matched_skills'] +
        [m['jd_skill'] for m in semantic['semantic_matches']]
    )


    total_jd_skills = len(jd_skills) if jd_skills else 1
    recall = len(all_matched) / total_jd_skills

    # Layer 3: TF-IDF
    tfidf_sim = compute_tfidf_similarity(
        cv_row.get('clean_text', ''),
        jd_row.get('clean_text', '')
    )

    # Final weighted score
    final_score = round(
        0.5 * recall +           # Skill coverage most important
        0.3 * tfidf_sim +        # Context relevance
        0.2 * exact['precision'] # Precision bonus
    , 4)

    return {
        'final_score'      : final_score,
        'exact_matches'    : exact['matched_skills'],
        'semantic_matches' : semantic['semantic_matches'],
        'true_missing'     : semantic['true_missing'],
        'tfidf_score'      : tfidf_sim,
        'recall'           : round(recall, 4),
        'precision'        : exact['precision']
    }


print(" Advanced matching function defined")
print("   - compute_semantic_skill_match()")
print("   - advanced_match()")

Loading SBERT model (reusing from earlier)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Advanced matching function defined
   - compute_semantic_skill_match()
   - advanced_match()


In [29]:

print("="*70)
print("ADVANCED MATCHING RESULTS (Exact + Semantic)")
print("="*70)

test_pairs = [
    ('Java Developer',   'javascript'),
    ('Python Developer', 'python'),
    ('DevOps',          'devops'),
    ('Data Science',    'big data'),
]

for cv_cat, jd_keyword in test_pairs:
    cv_row = df_cv_final[
        df_cv_final['job_category'] == cv_cat
    ].iloc[0].to_dict()

    jd_row = df_jd[
        df_jd['job_title'].str.lower().str.contains(jd_keyword, na=False)
    ].iloc[0].to_dict()

    result = advanced_match(cv_row, jd_row)

    print(f"\n Category : {cv_cat}")
    print(f"   JD Title : {jd_row.get('job_title', 'N/A')}")
    print(f"   {'─'*65}")
    print(f"    Final Score   : {result['final_score']:.3f}")
    print(f"    Recall        : {result['recall']:.3f}")
    print(f"    Precision     : {result['precision']:.3f}")
    print(f"    TF-IDF        : {result['tfidf_score']:.3f}")
    print(f"   {'─'*65}")

    print(f"    Exact Matches ({len(result['exact_matches'])}):")
    print(f"      {result['exact_matches'][:8]}")

    if result['semantic_matches']:
        print(f"\n    Semantic Matches ({len(result['semantic_matches'])}):")
        for match in result['semantic_matches'][:5]:
            print(f"      • '{match['cv_skill']}' → '{match['jd_skill']}' "
                  f"(sim: {match['similarity']:.2f})")

    print(f"\n    True Missing ({len(result['true_missing'])}):")
    print(f"      {result['true_missing'][:6]}")

# Comparing baseline vs advanced
print("\n" + "="*70)
print("BASELINE vs ADVANCED COMPARISON (DevOps example)")
print("="*70)

cv_row = df_cv_final[df_cv_final['job_category'] == 'DevOps'].iloc[0].to_dict()
jd_row = df_jd[df_jd['job_title'].str.lower().str.contains('devops', na=False)].iloc[0].to_dict()

baseline_result = baseline_match(cv_row, jd_row)
advanced_result = advanced_match(cv_row, jd_row)

print(f"\nBaseline:")
print(f"   Final Score : {baseline_result['final_score']:.3f}")
print(f"   Matched     : {len(baseline_result['matched_skills'])} skills")
print(f"   Missing     : {len(baseline_result['missing_skills'])} skills")

print(f"\nAdvanced (with SBERT):")
print(f"   Final Score : {advanced_result['final_score']:.3f}")
print(f"   Exact       : {len(advanced_result['exact_matches'])} skills")
print(f"   Semantic    : {len(advanced_result['semantic_matches'])} skills")
print(f"   Total Match : {len(advanced_result['exact_matches']) + len(advanced_result['semantic_matches'])} skills")
print(f"   True Missing: {len(advanced_result['true_missing'])} skills")

improvement = advanced_result['final_score'] - baseline_result['final_score']
print(f"\n Score Improvement: +{improvement:.3f} "
      f"({improvement/max(baseline_result['final_score'], 0.001)*100:.0f}%)")

ADVANCED MATCHING RESULTS (Exact + Semantic)

 Category : Java Developer
   JD Title : javascript developer
   ─────────────────────────────────────────────────────────────────
    Final Score   : 0.112
    Recall        : 0.133
    Precision     : 0.182
    TF-IDF        : 0.029
   ─────────────────────────────────────────────────────────────────
    Exact Matches (2):
      ['git', 'html']

    True Missing (13):
      ['adaptability', 'ajax', 'angular', 'communication', 'core javascript es6+', 'css']

 Category : Python Developer
   JD Title : python developer
   ─────────────────────────────────────────────────────────────────
    Final Score   : 0.144
    Recall        : 0.150
    Precision     : 0.273
    TF-IDF        : 0.048
   ─────────────────────────────────────────────────────────────────
    Exact Matches (3):
      ['git', 'python', 'sql']

    True Missing (17):
      ['communication', 'control flow', 'css', 'data types', 'django', 'exception handling']

 Category : DevO

In [30]:

# Testing different thresholds to find best spot
print("="*70)
print("THRESHOLD TUNING")
print("="*70)

cv_row = df_cv_final[df_cv_final['job_category'] == 'DevOps'].iloc[0].to_dict()
jd_row = df_jd[df_jd['job_title'].str.lower().str.contains('devops', na=False)].iloc[0].to_dict()

cv_skills = cv_row['normalized_skills_clean']
jd_skills = jd_row['ground_truth_skills_clean']

print(f"\nCV Skills: {cv_skills[:10]}")
print(f"JD Skills: {jd_skills}")

thresholds = [0.85, 0.80, 0.75, 0.70, 0.65, 0.60]

for thresh in thresholds:
    # Get exact matches first
    exact = compute_skill_overlap(cv_skills, jd_skills)

    # Try semantic on missing skills
    semantic = compute_semantic_skill_match(
        cv_skills,
        exact['missing_skills'],
        threshold=thresh
    )

    total_matched = len(exact['matched_skills']) + len(semantic['semantic_matches'])
    recall = total_matched / len(jd_skills)

    print(f"\n Threshold {thresh:.2f}")
    print(f"   Exact    : {len(exact['matched_skills'])}")
    print(f"   Semantic : {len(semantic['semantic_matches'])}")
    print(f"   Recall   : {recall:.3f}")

    if semantic['semantic_matches']:
        print(f"   Semantic matches:")
        for m in semantic['semantic_matches'][:3]:
            print(f"      • {m['cv_skill']:<20} → {m['jd_skill']:<30} ({m['similarity']:.2f})")

#  Test at optimal threshold
print("\n" + "="*70)
print("RETESTING WITH THRESHOLD = 0.65")
print("="*70)

test_pairs = [
    ('Java Developer',   'javascript'),
    ('Python Developer', 'python'),
    ('DevOps',          'devops'),
]

for cv_cat, jd_keyword in test_pairs:
    cv_row = df_cv_final[df_cv_final['job_category'] == cv_cat].iloc[0].to_dict()
    jd_row = df_jd[df_jd['job_title'].str.lower().str.contains(jd_keyword, na=False)].iloc[0].to_dict()

    cv_skills = cv_row['normalized_skills_clean']
    jd_skills = jd_row['ground_truth_skills_clean']

    exact = compute_skill_overlap(cv_skills, jd_skills)
    semantic = compute_semantic_skill_match(cv_skills, exact['missing_skills'], threshold=0.65)

    total_matched = len(exact['matched_skills']) + len(semantic['semantic_matches'])
    recall = total_matched / len(jd_skills)

    print(f"\n {cv_cat}")
    print(f"   Exact    : {len(exact['matched_skills'])} | {exact['matched_skills'][:5]}")
    print(f"   Semantic : {len(semantic['semantic_matches'])}")
    if semantic['semantic_matches']:
        for m in semantic['semantic_matches'][:3]:
            print(f"      • {m['cv_skill']} → {m['jd_skill']} ({m['similarity']:.2f})")
    print(f"   Recall   : {recall:.3f}")

THRESHOLD TUNING

CV Skills: ['automation', 'aws', 'docker', 'git', 'iam', 'jenkins', 'linux', 'networking', 'powershell', 'python']
JD Skills: ['aws', 'azure devops', 'ci/cd pipelines', 'terraform', 'docker', 'kubernetes', 'python']

 Threshold 0.85
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

 Threshold 0.80
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

 Threshold 0.75
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

 Threshold 0.70
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

 Threshold 0.65
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

 Threshold 0.60
   Exact    : 3
   Semantic : 0
   Recall   : 0.429

RETESTING WITH THRESHOLD = 0.65

 Java Developer
   Exact    : 2 | ['git', 'html']
   Semantic : 0
   Recall   : 0.133

 Python Developer
   Exact    : 3 | ['git', 'python', 'sql']
   Semantic : 0
   Recall   : 0.150

 DevOps
   Exact    : 3 | ['aws', 'docker', 'python']
   Semantic : 0
   Recall   : 0.429


In [31]:

import pickle

save_dir = '/content/drive/MyDrive/skill_match_with_JD/'

#  Save matching functions as module
matching_code = '''
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Load SBERT once
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def clean_skill_string(skill):
    if not isinstance(skill, str):
        return ''
    s = skill.lower().strip()
    s = re.sub(r'\\(.*?\\)', '', s)
    s = re.sub(r'[^a-z0-9\\s\\+\\#\\.\\-\\/]', '', s)
    s = re.sub(r'\\s+', ' ', s).strip()
    return s

def normalize_skill_list(skill_list):
    if not isinstance(skill_list, list):
        return []
    cleaned = [clean_skill_string(s) for s in skill_list]
    return [s for s in cleaned if len(s) > 1]

def compute_skill_overlap(cv_skills, jd_skills):
    if not cv_skills or not jd_skills:
        return {'overlap_score': 0.0, 'matched_skills': [], 'missing_skills': [], 'extra_skills': []}

    cv_set = set([s.lower().strip() for s in cv_skills])
    jd_set = set([s.lower().strip() for s in jd_skills])

    matched = cv_set & jd_set
    missing = jd_set - cv_set
    extra = cv_set - jd_set

    precision = len(matched) / len(cv_set) if cv_set else 0
    recall = len(matched) / len(jd_set) if jd_set else 0
    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0)

    return {
        'overlap_score': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'matched_skills': sorted(list(matched)),
        'missing_skills': sorted(list(missing)),
        'extra_skills': sorted(list(extra))
    }

def compute_tfidf_similarity(cv_text, jd_text):
    if not cv_text or not jd_text:
        return 0.0

    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000)

    try:
        tfidf_matrix = vectorizer.fit_transform([cv_text, jd_text])
        sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(sim), 4)
    except:
        return 0.0

def match_cv_to_jd(cv_skills, jd_skills, cv_text='', jd_text=''):

    # Clean inputs
    cv_skills_clean = normalize_skill_list(cv_skills)
    jd_skills_clean = normalize_skill_list(jd_skills)

    # Skill overlap
    overlap = compute_skill_overlap(cv_skills_clean, jd_skills_clean)

    # TF-IDF if text provided
    tfidf_score = compute_tfidf_similarity(cv_text, jd_text) if cv_text and jd_text else 0.0

    # Final weighted score
    final_score = round(0.7 * overlap['recall'] + 0.3 * tfidf_score, 4)

    match_percentage = round(overlap['recall'] * 100, 1)

    return {
        'final_score': final_score,
        'match_percentage': match_percentage,
        'matched_skills': overlap['matched_skills'],
        'missing_skills': overlap['missing_skills'],
        'extra_skills': overlap['extra_skills'],
        'precision': overlap['precision'],
        'recall': overlap['recall'],
        'tfidf_score': tfidf_score
    }
'''

with open(f'{save_dir}matching_engine.py', 'w') as f:
    f.write(matching_code)

print(" Saved matching_engine.py")

# Save datasets
df_cv_final[['cv_id', 'job_category', 'normalized_skills_clean', 'clean_text']].to_json(
    f'{save_dir}df_cv_production.jsonl', orient='records', lines=True
)

df_jd[['job_id', 'job_title', 'ground_truth_skills_clean', 'clean_text']].to_json(
    f'{save_dir}df_jd_production.jsonl', orient='records', lines=True
)

print("Saved production datasets")

# Test saved module
print("\n" + "="*60)
print("TESTING SAVED MATCHING ENGINE")
print("="*60)

exec(open(f'{save_dir}matching_engine.py').read())

# Test case
cv_skills = ['python', 'sql', 'git', 'docker', 'aws', 'linux']
jd_skills = ['python', 'sql', 'postgresql', 'docker', 'kubernetes', 'aws', 'terraform']

result = match_cv_to_jd(cv_skills, jd_skills)

print(f"\nTest Match:")
print(f"   Final Score       : {result['final_score']:.3f}")
print(f"   Match Percentage  : {result['match_percentage']}%")
print(f"   Matched Skills    : {result['matched_skills']}")
print(f"   Missing Skills    : {result['missing_skills']}")
print(f"   Extra Skills      : {result['extra_skills']}")

print(f"\n Matching engine is production-ready")

 Saved matching_engine.py
Saved production datasets

TESTING SAVED MATCHING ENGINE


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Test Match:
   Final Score       : 0.400
   Match Percentage  : 57.1%
   Matched Skills    : ['aws', 'docker', 'python', 'sql']
   Missing Skills    : ['kubernetes', 'postgresql', 'terraform']
   Extra Skills      : ['git', 'linux']

 Matching engine is production-ready


test ui*********************************************************************

In [32]:

!pip install PyPDF2 gradio -q

print(" Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 1.3 MB/s eta 0:00:00
 Dependencies installed


In [33]:
import sys
sys.path.append('/content/drive/MyDrive/skill_match_with_JD/')


exec(open('/content/drive/MyDrive/skill_match_with_JD/matching_engine.py').read())

print(" Matching engine loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Matching engine loaded


In [34]:
import PyPDF2
import re

def extract_text_from_pdf(pdf_file):

    try:
        reader = PyPDF2.PdfReader(pdf_file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
        return text
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

def clean_text(text):

    text = text.lower()
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[\+\(]?[0-9][0-9\s\-\(\)]{8,}[0-9]', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\b\d{1,2}[\/\-]\d{4}\b', '', text)
    text = re.sub(r'[^a-z0-9\s\.\+\#\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(" PDF extraction functions ready")

 PDF extraction functions ready


In [35]:
import pickle


with open('/content/drive/MyDrive/skill_match_with_JD/title_to_skills.pkl', 'rb') as f:
    title_to_skills = pickle.load(f)


all_known_skills = set()
for skills_list in title_to_skills.values():
    all_known_skills.update([s.lower() for s in skills_list])

print(f"Loaded {len(all_known_skills)} known skills from ontology")

def extract_skills_from_text(text, known_skills):

    text_lower = text.lower()
    found_skills = []

    for skill in known_skills:
        if len(skill) > 2 and re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found_skills.append(skill)

    return sorted(list(set(found_skills)))

print(" Skill extraction function ready")

Loaded 581 known skills from ontology
 Skill extraction function ready


In [36]:
import gradio as gr

def match_cv_jd_demo(cv_pdf, jd_pdf):
    """
    Main demo function that processes CV and JD PDFs
    and returns matching results
    """
    # Extract text from PDFs
    cv_text = extract_text_from_pdf(cv_pdf)
    jd_text = extract_text_from_pdf(jd_pdf)

    if "Error" in cv_text or "Error" in jd_text:
        return cv_text, jd_text, "Error processing PDFs"

    # Clean text
    cv_clean = clean_text(cv_text)
    jd_clean = clean_text(jd_text)

    # Extract skills
    cv_skills = extract_skills_from_text(cv_clean, all_known_skills)
    jd_skills = extract_skills_from_text(jd_clean, all_known_skills)

    # Run matching
    result = match_cv_to_jd(cv_skills, jd_skills, cv_clean, jd_clean)

    # Format output
    output = f"""
 **MATCHING RESULTS**
═══════════════════════════════════════════════════════════

 **MATCH SCORE: {result['final_score']:.3f} ({result['match_percentage']}%)**

 **Metrics:**
   • Recall (Coverage):  {result['recall']:.1%}
   • Precision:          {result['precision']:.1%}
   • Text Similarity:    {result['tfidf_score']:.1%}

═══════════════════════════════════════════════════════════

 **CV SKILLS ({len(cv_skills)}):**
{', '.join(cv_skills[:30])}
{'...' if len(cv_skills) > 30 else ''}

 **JD REQUIRED SKILLS ({len(jd_skills)}):**
{', '.join(jd_skills[:30])}
{'...' if len(jd_skills) > 30 else ''}

═══════════════════════════════════════════════════════════

 **MATCHED SKILLS ({len(result['matched_skills'])}):**
{', '.join(result['matched_skills']) if result['matched_skills'] else 'None'}

 **MISSING SKILLS ({len(result['missing_skills'])}):**
{', '.join(result['missing_skills']) if result['missing_skills'] else 'None'}

 **EXTRA SKILLS IN CV ({len(result['extra_skills'])}):**
{', '.join(result['extra_skills'][:20]) if result['extra_skills'] else 'None'}
{'...' if len(result['extra_skills']) > 20 else ''}

═══════════════════════════════════════════════════════════

 **Interpretation:**
- Match % shows how many JD skills the candidate has
- Missing skills are gaps the candidate needs to fill
- Extra skills show additional qualifications beyond requirements
"""

    return output

# Create Gradio interface
demo = gr.Interface(
    fn=match_cv_jd_demo,
    inputs=[
        gr.File(label="📄 Upload CV (PDF)", file_types=[".pdf"]),
        gr.File(label="📋 Upload Job Description (PDF)", file_types=[".pdf"])
    ],
    outputs=gr.Textbox(label="Matching Results", lines=30),
    title=" CV-JD Matching System Demo",
    description="""
    Upload a CV and Job Description PDF to see:
    - Match score and percentage
    - Matched skills
    - Missing skills
    - Extra skills candidate has
    """,
    examples=[]
)

# Launch
demo.launch(share=True, debug=True)




Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f5a8e8188ce3fc9151.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f5a8e8188ce3fc9151.gradio.live
